# 07c GLORYS-cell Moran's I sensitivity

**Manuscript outputs:** Supporting sensitivity analyses for Section 4.2 (cell level)

**Inputs:** 47-cell table and spatial weights from stage 07a

Inputs are read from `Master_raw/`; everything produced is written to `Outputs/` (manuscript figures in `Outputs/Figures`, tables in `Outputs/Tables`). The first code cell lists every path. Run the cells in order.

# 02 | Geostatistical analysis — Stage 14 onwards

Run Notebook 01 first. This notebook reloads exported outputs independently and uses identical cell order for Queen and Rook. Inferential sample size is 47 native cells, not 12,452 transects. Permutation inference and choice of spatial weights require sensitivity analysis; do not reuse superseded transect-level or interpolated-SLA inferential statistics.

In [1]:
from pathlib import Path

# ------------------------------------------------------------
# REPOSITORY ROOT: first folder above this notebook that contains Master_raw
# (e.g. D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024)
# ------------------------------------------------------------
_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents) if (p / "Master_raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No Master_raw folder found above {_here}. Open this notebook inside the repository.")
RAW = ROOT / "Master_raw"          # inputs
OUTPUTS = ROOT / "Outputs"         # everything this repository produces
FIGURES = OUTPUTS / "Figures"      # manuscript figures
TABLES = OUTPUTS / "Tables"        # manuscript tables
TEXT_STATS = TABLES / "Text_statistics"
for _d in (FIGURES, TABLES, TEXT_STATS):
    _d.mkdir(parents=True, exist_ok=True)
OVERWRITE = True                   # re-running replaces this notebook's earlier outputs


def first_existing(*candidates):
    """Return the first path that exists (otherwise the first one)."""
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return Path(candidates[0])


def save_table(df, name, index=False, folder=None):
    path = (folder or TABLES) / f"{name}.csv"
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("Saved:", path.relative_to(ROOT))


def save_figure(fig, name, dpi=600):
    for ext in ("jpg",):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=dpi, bbox_inches="tight", pil_kwargs=({"quality": 95} if ext == "jpg" else None))
    print("Saved:", (FIGURES / f"{name}.jpg").relative_to(ROOT))


def remove_existing(*files):
    for f in map(Path, files):
        for t in (list(f.parent.glob(f.stem + ".*")) if f.suffix == ".shp" else [f]):
            if t.exists():
                if not OVERWRITE:
                    raise FileExistsError(t)
                t.unlink()

# ------------------------------------------------------------
# INPUTS AND OUTPUT FOLDERS FOR THIS NOTEBOOK
# ------------------------------------------------------------
GEOSTATS_DIR = OUTPUTS / "Geostatistics"
GEOSTATS_CELL_DIR = GEOSTATS_DIR / "GLORYS_Cell_Level"

print("Repository root:", ROOT)
for _label, _p in [("GEOSTATS_CELL_DIR", GEOSTATS_CELL_DIR)]:
    print(f"  {'ok     ' if _p.exists() else 'MISSING'} {_label}: {_p.relative_to(ROOT)}")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
CELL_DIR = GEOSTATS_CELL_DIR
analysis_grid=gpd.read_file(CELL_DIR/"glorys_47_cells_QC_verified.gpkg",layer="glorys_47_cells")
cell_ids=pd.read_csv(CELL_DIR/"glorys_47_cell_order.csv")["Pixel_ID"].astype(str).tolist()
assert len(cell_ids)==47 and len(set(cell_ids))==47
analysis_grid=analysis_grid.set_index("Pixel_ID").loc[cell_ids].reset_index()
def load_weights(name):
    b=pd.read_csv(CELL_DIR/f"glorys_{name}_weights_binary.csv",index_col=0).loc[cell_ids,cell_ids].to_numpy(dtype=float)
    w=pd.read_csv(CELL_DIR/f"glorys_{name}_weights_row_standardised.csv",index_col=0).loc[cell_ids,cell_ids].to_numpy(dtype=float)
    assert np.array_equal(b,b.T) and np.allclose(w.sum(axis=1),1)
    return b,w
queen_binary,queen_W=load_weights("queen")
rook_binary,rook_W=load_weights("rook")
assert int(queen_binary.sum()/2)==73 and int(rook_binary.sum()/2)==45
print("Ready: 47 ordered GLORYS cells; Queen/Rook weights validated.")

Repository root: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024
  ok      GEOSTATS_CELL_DIR: Outputs\Geostatistics\GLORYS_Cell_Level
Ready: 47 ordered GLORYS cells; Queen/Rook weights validated.


## Stage 14 | Global Moran’s I: median WLR, Queen versus Rook

Two-sided permutation tests compare deviations from each weight matrix’s empirical permutation-null mean. This is a global unrestricted-label randomisation diagnostic; the disconnected Rook components motivate a further component-aware sensitivity analysis. No results should be entered in the thesis until executed and checked.

In [2]:
def morans_i(values,W):
    z=np.asarray(values,dtype=float)-np.mean(values)
    return len(z)/W.sum() * (z@W@z)/(z@z)
def permutation_moran(values,W,permutations=9999,seed=42):
    rng=np.random.default_rng(seed)
    observed=morans_i(values,W)
    simulated=np.array([morans_i(rng.permutation(values),W) for _ in range(permutations)])
    centre=simulated.mean()
    p=(1+np.count_nonzero(np.abs(simulated-centre)>=abs(observed-centre)))/(permutations+1)
    return observed,centre,simulated.std(ddof=1),p
y=pd.to_numeric(analysis_grid["WLR_Median"],errors="coerce").to_numpy(dtype=float)
assert np.isfinite(y).all() and np.var(y)>0
rows=[]
for label,W in [("Queen",queen_W),("Rook",rook_W)]:
    observed,null_mean,null_sd,p=permutation_moran(y,W)
    rows.append(dict(Weights=label,Cells=len(y),Moran_I=observed,Permutation_Null_Mean=null_mean,Permutation_SD=null_sd,P_Two_Sided=p,Permutations=9999))
moran_comparison=pd.DataFrame(rows)
display(moran_comparison.round(5))
moran_comparison.to_csv(CELL_DIR/"stage14_global_moran_weights_sensitivity.csv",index=False)

,Weights,Cells,Moran_I,Permutation_Null_Mean,Permutation_SD,P_Two_Sided,Permutations
0,Queen,47,0.14442,-0.02206,0.10013,0.0965,9999
1,Rook,47,0.20404,-0.02122,0.12673,0.0869,9999


## Planned analyses (not executed)

1. Repeat global Moran’s I with cell mean WLR, and examine influence of cells with very small valid-transect counts.
2. Evaluate component-aware inference for the three disconnected Rook components, documenting its conditional null hypothesis.
3. Only then run local Moran/LISA, label non-significant cells explicitly, and apply an FDR procedure for multiple local tests.
4. Fit parsimonious cell-level associations of WLR with GLORYS trend and variability; check multicollinearity, residual spatial autocorrelation, influential cells, and sample-size limitations.
5. Compare specifications and uncertainty, not just p-values. Do not interpret cross-sectional association as causality, nor fit GWR automatically with n=47.
6. Transfer only verified numerical results into Chapter Four and update abstract and conclusions.

In [3]:
# ============================================================
# STAGE 15 FIX: RECONSTRUCT ROOK CONNECTED COMPONENTS
# Uses the existing Rook weights matrix and cell ordering
# ============================================================

import numpy as np

# Validate the inputs already loaded by the geostatistics notebook.
assert rook_W.shape == (len(cell_ids), len(cell_ids))
assert len(cell_ids) == len(set(cell_ids))

# Convert row-standardised weights to a binary adjacency matrix.
adjacency = np.asarray(rook_W) > 0

# Verify that the underlying neighbour relationships are symmetric.
assert np.array_equal(adjacency, adjacency.T)
assert not np.any(np.diag(adjacency))

visited = set()
rook_components = []

for start in range(len(cell_ids)):

    if start in visited:
        continue

    stack = [start]
    component = []

    while stack:

        current = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        component.append(cell_ids[current])

        adjacent = np.flatnonzero(
            adjacency[current]
        )

        stack.extend(
            int(i) for i in adjacent
            if int(i) not in visited
        )

    rook_components.append(component)

# Sort components from largest to smallest.
rook_components.sort(
    key=len,
    reverse=True
)

print("ROOK COMPONENT VALIDATION")
print("Number of components:", len(rook_components))
print("Component sizes:", [len(c) for c in rook_components])
print("Total cells:", sum(len(c) for c in rook_components))

assert len(rook_components) == 3
assert sorted(
    [len(c) for c in rook_components],
    reverse=True
) == [38, 5, 4]

print("\nValidation passed. Proceed to Stage 15.")

ROOK COMPONENT VALIDATION
Number of components: 3
Component sizes: [38, 5, 4]
Total cells: 47

Validation passed. Proceed to Stage 15.


In [4]:
# ============================================================
# STAGE 15: COMPONENT-RESTRICTED MORAN'S I
# Run AFTER the Rook component validation cell
# ============================================================

import numpy as np
import pandas as pd

# 1. Align WLR values with the spatial weights matrix.
ordered = analysis_grid.set_index("Pixel_ID").loc[cell_ids]

y = pd.to_numeric(
    ordered["WLR_Median"],
    errors="coerce"
).to_numpy(dtype=float)

assert np.isfinite(y).all()
assert len(y) == 47

# 2. Convert component cell IDs into matrix positions.
id_to_index = {
    cell_id: i
    for i, cell_id in enumerate(cell_ids)
}

component_indices = [
    np.array([id_to_index[cell] for cell in component], dtype=int)
    for component in rook_components
]

assert sorted(
    [len(component) for component in component_indices],
    reverse=True
) == [38, 5, 4]

# 3. Define Moran's I.
def morans_i(values, W):
    z = values - np.mean(values)
    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / (z @ z)
    )

# 4. Calculate observed Moran's I.
observed = morans_i(y, rook_W)

# 5. Permute values only within each Rook component.
rng = np.random.default_rng(42)
n_permutations = 9999

simulated = np.empty(n_permutations)

for i in range(n_permutations):

    permuted = y.copy()

    for indices in component_indices:
        permuted[indices] = rng.permutation(y[indices])

    simulated[i] = morans_i(permuted, rook_W)

# 6. Two-sided permutation p-value centred on the simulated null mean.
null_mean = simulated.mean()

p_value = (
    1 + np.sum(
        np.abs(simulated - null_mean)
        >= abs(observed - null_mean)
    )
) / (n_permutations + 1)

# 7. Create Table 18.
table18 = pd.DataFrame([{
    "Weights": "Rook",
    "Permutation_Method": "Within Rook components",
    "Cells": len(y),
    "Components": len(component_indices),
    "Moran_I": observed,
    "Permutation_Null_Mean": null_mean,
    "Permutation_SD": simulated.std(ddof=1),
    "P_Two_Sided": p_value,
    "Permutations": n_permutations
}])

print("\nTABLE 18: COMPONENT-RESTRICTED MORAN'S I")
print(table18.round(5).to_string(index=False))

# 8. Save results.
table18.to_csv(
    CELL_DIR / "stage15_moran_component_sensitivity.csv",
    index=False
)

print("\nStage 15 completed successfully.")


TABLE 18: COMPONENT-RESTRICTED MORAN'S I
Weights     Permutation_Method  Cells  Components  Moran_I  Permutation_Null_Mean  Permutation_SD  P_Two_Sided  Permutations
   Rook Within Rook components     47           3  0.20404               -0.08739         0.12058       0.0205          9999

Stage 15 completed successfully.


In [5]:
# ============================================================
# STAGE 16A: CELL-LEVEL WLR SUMMARY AND SAMPLE-SIZE AUDIT
#
# Objectives:
# 1. Verify the 47-cell analysis dataset.
# 2. Compare mean and median WLR.
# 3. Identify sparsely sampled cells.
# 4. Flag large mean-median differences for investigation.
#
# No cells or transects are excluded.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Confirm required inputs
# ------------------------------------------------------------

assert "analysis_grid" in globals()
assert "cell_ids" in globals()
assert "CELL_DIR" in globals()

# Ensure every cell is represented exactly once.
assert analysis_grid["Pixel_ID"].is_unique
assert len(analysis_grid) == 47

# Align cell order with the spatial weights matrices.
cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .reset_index()
    .copy()
)

# ------------------------------------------------------------
# 2. Locate the required columns
# ------------------------------------------------------------

print("Available columns:")
print(cells.columns.tolist())

def find_column(columns, candidates, description):

    for candidate in candidates:
        if candidate in columns:
            return candidate

    raise KeyError(
        f"Could not find {description}. "
        f"Available columns: {list(columns)}"
    )

count_col = find_column(
    cells.columns,
    [
        "WLR_Count",
        "WLR_Valid_Count",
        "Valid_WLR_Count",
        "N_Transects",
        "Transect_Count",
        "n_transects",
        "n_valid"
    ],
    "valid WLR transect count"
)

mean_col = find_column(
    cells.columns,
    [
        "WLR_Mean",
        "Mean_WLR",
        "WLR_mean"
    ],
    "mean WLR"
)

median_col = find_column(
    cells.columns,
    [
        "WLR_Median",
        "Median_WLR",
        "WLR_median"
    ],
    "median WLR"
)

print("\nSelected columns:")
print("Count:", count_col)
print("Mean:", mean_col)
print("Median:", median_col)

# ------------------------------------------------------------
# 3. Construct the audit table
# ------------------------------------------------------------

audit = pd.DataFrame({
    "Pixel_ID": cells["Pixel_ID"].astype(str),
    "Valid_Transects": pd.to_numeric(
        cells[count_col],
        errors="coerce"
    ),
    "WLR_Mean": pd.to_numeric(
        cells[mean_col],
        errors="coerce"
    ),
    "WLR_Median": pd.to_numeric(
        cells[median_col],
        errors="coerce"
    )
})

# Verify the expected cell coverage and valid observations.
assert len(audit) == 47
assert audit["Valid_Transects"].notna().all()
assert audit["Valid_Transects"].gt(0).all()
assert audit[["WLR_Mean", "WLR_Median"]].notna().all().all()

print("\nVALIDATION")
print("Cells:", len(audit))
print(
    "Total valid WLR transects:",
    int(audit["Valid_Transects"].sum())
)

# Expected total following the three invalid regression records.
assert int(audit["Valid_Transects"].sum()) == 12452

# ------------------------------------------------------------
# 4. Compare mean and median WLR
# ------------------------------------------------------------

audit["Mean_Minus_Median"] = (
    audit["WLR_Mean"] - audit["WLR_Median"]
)

audit["Absolute_Difference"] = (
    audit["Mean_Minus_Median"].abs()
)

# Descriptive flags only. These are NOT exclusion criteria.
audit["Fewer_Than_10"] = (
    audit["Valid_Transects"] < 10
)

audit["Fewer_Than_30"] = (
    audit["Valid_Transects"] < 30
)

# Rank cells by absolute mean-median difference.
audit = audit.sort_values(
    "Absolute_Difference",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Table 19: Overall audit
# ------------------------------------------------------------

table19 = pd.DataFrame([{
    "Cells": len(audit),
    "Valid_Transects": int(
        audit["Valid_Transects"].sum()
    ),
    "Min_Transects_Per_Cell": int(
        audit["Valid_Transects"].min()
    ),
    "Median_Transects_Per_Cell": float(
        audit["Valid_Transects"].median()
    ),
    "Max_Transects_Per_Cell": int(
        audit["Valid_Transects"].max()
    ),
    "Cells_N_Less_10": int(
        audit["Fewer_Than_10"].sum()
    ),
    "Cells_N_Less_30": int(
        audit["Fewer_Than_30"].sum()
    ),
    "Mean_Of_Cell_Means": float(
        audit["WLR_Mean"].mean()
    ),
    "Mean_Of_Cell_Medians": float(
        audit["WLR_Median"].mean()
    )
}])

print("\nTABLE 19: CELL-LEVEL SUMMARY AUDIT")
print(table19.round(4).to_string(index=False))

# ------------------------------------------------------------
# 6. Table 20: Largest mean-median differences
# ------------------------------------------------------------

print("\nTABLE 20: TEN LARGEST MEAN-MEDIAN DIFFERENCES")

print(
    audit[
        [
            "Pixel_ID",
            "Valid_Transects",
            "WLR_Mean",
            "WLR_Median",
            "Mean_Minus_Median",
            "Absolute_Difference"
        ]
    ]
    .head(10)
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Table 21: Sparsely sampled cells
# ------------------------------------------------------------

print("\nTABLE 21: CELLS WITH FEWER THAN 30 TRANSECTS")

sparse = (
    audit.loc[
        audit["Fewer_Than_30"],
        [
            "Pixel_ID",
            "Valid_Transects",
            "WLR_Mean",
            "WLR_Median",
            "Absolute_Difference"
        ]
    ]
    .sort_values("Valid_Transects")
)

if sparse.empty:
    print("No cells have fewer than 30 transects.")
else:
    print(
        sparse.round(4).to_string(index=False)
    )

# ------------------------------------------------------------
# 8. Save reproducible outputs
# ------------------------------------------------------------

audit.to_csv(
    CELL_DIR / "stage16a_cell_wlr_summary_audit.csv",
    index=False
)

table19.to_csv(
    CELL_DIR / "stage16a_overall_summary.csv",
    index=False
)

sparse.to_csv(
    CELL_DIR / "stage16a_sparse_cells.csv",
    index=False
)

print("\nStage 16A completed successfully.")
print("No cells or transects were excluded.")

Available columns:
['Pixel_ID', 'Grid_Row', 'Grid_Col', 'Grid_Lon', 'Grid_Lat', 'SL_Trend_mm_yr', 'SL_Variability_m', 'Transect_Count', 'Valid_WLR_Count', 'Invalid_WLR_Count', 'IQR_Flag_Count', 'WLR_Mean', 'WLR_Median', 'WLR_SD', 'WLR_Min', 'WLR_Max', 'Transect_Mean_Longitude', 'Transect_Mean_Latitude', 'Zones', 'IQR_Flag_Percent', 'Mean_Median_Difference', 'Queen_Neighbour_Count', 'geometry']

Selected columns:
Count: Valid_WLR_Count
Mean: WLR_Mean
Median: WLR_Median

VALIDATION
Cells: 47
Total valid WLR transects: 12452

TABLE 19: CELL-LEVEL SUMMARY AUDIT
 Cells  Valid_Transects  Min_Transects_Per_Cell  Median_Transects_Per_Cell  Max_Transects_Per_Cell  Cells_N_Less_10  Cells_N_Less_30  Mean_Of_Cell_Means  Mean_Of_Cell_Medians
    47            12452                       3                      297.0                     555                2                2              0.2582                0.0178

TABLE 20: TEN LARGEST MEAN-MEDIAN DIFFERENCES
 Pixel_ID  Valid_Transects  WLR_Mean  W

In [6]:
# ============================================================
# STAGE 16B: MEAN VS MEDIAN WLR - GLOBAL MORAN'S I
#
# Purpose:
# Compare spatial autocorrelation using cell mean and median WLR.
#
# Design:
# - All 47 cells retained.
# - Queen and Rook row-standardised weights.
# - 9,999 unrestricted permutations.
# - Same random seed and two-sided test as Stage 14.
#
# No cells or transects are excluded.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

required = [
    "analysis_grid",
    "cell_ids",
    "queen_W",
    "rook_W",
    "CELL_DIR"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required objects: " + ", ".join(missing)
    )

# ------------------------------------------------------------
# 2. Align observations with weights
# ------------------------------------------------------------

assert analysis_grid["Pixel_ID"].is_unique
assert len(cell_ids) == 47

ordered = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
)

responses = {
    "Median_WLR": pd.to_numeric(
        ordered["WLR_Median"],
        errors="coerce"
    ).to_numpy(dtype=float),

    "Mean_WLR": pd.to_numeric(
        ordered["WLR_Mean"],
        errors="coerce"
    ).to_numpy(dtype=float)
}

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

for name, values in responses.items():
    assert len(values) == 47, name
    assert np.isfinite(values).all(), name
    assert np.var(values) > 0, name

for name, W in weights.items():
    assert W.shape == (47, 47), name
    assert np.allclose(W.sum(axis=1), 1), name
    assert np.allclose(np.diag(W), 0), name

# ------------------------------------------------------------
# 3. Moran's I
# ------------------------------------------------------------

def calculate_morans_i(values, W):

    z = values - values.mean()

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / (z @ z)
    )

# ------------------------------------------------------------
# 4. Unrestricted permutation test
# ------------------------------------------------------------

def unrestricted_moran_test(
    values,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    observed = calculate_morans_i(values, W)

    simulated = np.empty(permutations)

    for i in range(permutations):

        permuted = rng.permutation(values)

        simulated[i] = calculate_morans_i(
            permuted,
            W
        )

    null_mean = simulated.mean()

    p_two_sided = (
        1 + np.count_nonzero(
            np.abs(simulated - null_mean)
            >= abs(observed - null_mean)
        )
    ) / (permutations + 1)

    return {
        "Moran_I": observed,
        "Permutation_Null_Mean": null_mean,
        "Permutation_SD": simulated.std(ddof=1),
        "P_Two_Sided": p_two_sided,
        "Permutations": permutations
    }

# ------------------------------------------------------------
# 5. Run four combinations
# ------------------------------------------------------------

records = []

for response_name, values in responses.items():

    for weights_name, W in weights.items():

        result = unrestricted_moran_test(
            values=values,
            W=W,
            permutations=9999,
            seed=42
        )

        records.append({
            "Response": response_name,
            "Weights": weights_name,
            "Cells": len(values),
            **result
        })

table22 = pd.DataFrame(records)

# ------------------------------------------------------------
# 6. Validate reproduction of Stage 14
# ------------------------------------------------------------

median_results = table22.set_index(
    ["Response", "Weights"]
)

assert np.isclose(
    median_results.loc[
        ("Median_WLR", "Queen"), "Moran_I"
    ],
    0.14442,
    atol=0.00001
), "Queen median Moran's I differs from Stage 14."

assert np.isclose(
    median_results.loc[
        ("Median_WLR", "Rook"), "Moran_I"
    ],
    0.20404,
    atol=0.00001
), "Rook median Moran's I differs from Stage 14."

# ------------------------------------------------------------
# 7. Display and save
# ------------------------------------------------------------

print("\nTABLE 22: MEAN VS MEDIAN WLR MORAN'S I")

print(
    table22.round(5).to_string(index=False)
)

table22.to_csv(
    CELL_DIR / "stage16b_mean_median_moran_comparison.csv",
    index=False
)

print("\nStage 16B completed successfully.")
print("Stage 14 observed Moran's I values reproduced.")
print("All 47 cells retained.")


TABLE 22: MEAN VS MEDIAN WLR MORAN'S I
  Response Weights  Cells  Moran_I  Permutation_Null_Mean  Permutation_SD  P_Two_Sided  Permutations
Median_WLR   Queen     47  0.14442               -0.02206         0.10013       0.0965          9999
Median_WLR    Rook     47  0.20404               -0.02122         0.12673       0.0869          9999
  Mean_WLR   Queen     47  0.04986               -0.02179         0.10092       0.4027          9999
  Mean_WLR    Rook     47  0.04613               -0.02210         0.13006       0.4465          9999

Stage 16B completed successfully.
Stage 14 observed Moran's I values reproduced.
All 47 cells retained.


In [7]:
# ============================================================
# STAGE 16C: SPARSE-CELL SENSITIVITY ANALYSIS
#
# Purpose:
# Test whether cells with fewer than 30 valid WLR transects
# influence global Moran's I.
#
# Primary response: median WLR
# Secondary response: mean WLR
#
# Full dataset remains unchanged.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Align cell attributes and weights
# ------------------------------------------------------------

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

assert len(cells) == 47

counts = pd.to_numeric(
    cells["Valid_WLR_Count"],
    errors="coerce"
).to_numpy()

assert np.isfinite(counts).all()
assert int(counts.sum()) == 12452

# ------------------------------------------------------------
# 2. Identify retained and excluded cells
# ------------------------------------------------------------

threshold = 30

keep = counts >= threshold
remove = ~keep

removed_ids = np.array(cell_ids)[remove].tolist()

print("STAGE 16C: SAMPLE-SIZE AUDIT")
print("Original cells:", len(cell_ids))
print("Retained cells:", int(keep.sum()))
print("Temporarily excluded cells:", int(remove.sum()))
print("Excluded Pixel_IDs:", removed_ids)

assert int(keep.sum()) == 45
assert set(removed_ids) == {
    "R022_C015",
    "R015_C012"
}

# ------------------------------------------------------------
# 3. Reconstruct weights after subsetting
# ------------------------------------------------------------

def subset_weights(W, keep_mask):

    # Recover binary neighbour relationships.
    binary = (np.asarray(W) > 0).astype(float)

    # Retain relationships between remaining cells.
    reduced = binary[np.ix_(keep_mask, keep_mask)]

    neighbour_counts = reduced.sum(axis=1)

    # Identify isolates before standardisation.
    isolates = np.flatnonzero(neighbour_counts == 0)

    # Row-standardise cells with at least one neighbour.
    standardised = np.zeros_like(reduced)

    nonisolated = neighbour_counts > 0

    standardised[nonisolated] = (
        reduced[nonisolated]
        / neighbour_counts[nonisolated, None]
    )

    return standardised, neighbour_counts, isolates

reduced_weights = {}

retained_ids = np.array(cell_ids)[keep]

for name, W in {
    "Queen": queen_W,
    "Rook": rook_W
}.items():

    W_reduced, neighbour_counts, isolates = (
        subset_weights(W, keep)
    )

    reduced_weights[name] = W_reduced

    print(f"\n{name} weights:")
    print("Remaining neighbour links:",
          int((W_reduced > 0).sum() / 2))
    print("Isolated cells:",
          retained_ids[isolates].tolist())

    # Stop if removal creates isolates. We should then
    # review the graph before selecting a test convention.
    if len(isolates) > 0:
        raise RuntimeError(
            f"{name} has isolated cells after subsetting. "
            "Review the reported IDs before continuing."
        )

# ------------------------------------------------------------
# 4. Moran's I
# ------------------------------------------------------------

def morans_i_16c(values, W):

    z = values - values.mean()

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / (z @ z)
    )

# ------------------------------------------------------------
# 5. Unrestricted permutation test
# ------------------------------------------------------------

def permutation_test_16c(
    values,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    observed = morans_i_16c(values, W)

    simulated = np.empty(permutations)

    for i in range(permutations):

        simulated[i] = morans_i_16c(
            rng.permutation(values),
            W
        )

    null_mean = simulated.mean()

    p_value = (
        1 + np.count_nonzero(
            np.abs(simulated - null_mean)
            >= abs(observed - null_mean)
        )
    ) / (permutations + 1)

    return {
        "Moran_I": observed,
        "Null_Mean": null_mean,
        "Null_SD": simulated.std(ddof=1),
        "P_Two_Sided": p_value
    }

# ------------------------------------------------------------
# 6. Run sensitivity tests
# ------------------------------------------------------------

records = []

for response_name, column in {
    "Median_WLR": "WLR_Median",
    "Mean_WLR": "WLR_Mean"
}.items():

    full_values = pd.to_numeric(
        cells[column],
        errors="coerce"
    ).to_numpy(dtype=float)

    assert np.isfinite(full_values).all()

    reduced_values = full_values[keep]

    for weights_name, full_W in {
        "Queen": queen_W,
        "Rook": rook_W
    }.items():

        # Full 47-cell result.
        full_result = permutation_test_16c(
            full_values,
            full_W
        )

        # Reduced 45-cell result.
        reduced_result = permutation_test_16c(
            reduced_values,
            reduced_weights[weights_name]
        )

        records.append({
            "Response": response_name,
            "Weights": weights_name,
            "Cells": 47,
            "Sample": "All cells",
            **full_result
        })

        records.append({
            "Response": response_name,
            "Weights": weights_name,
            "Cells": 45,
            "Sample": "Exclude N < 30",
            **reduced_result
        })

# ------------------------------------------------------------
# 7. Display Table 23
# ------------------------------------------------------------

table23 = pd.DataFrame(records)

print("\nTABLE 23: SPARSE-CELL SENSITIVITY")

print(
    table23.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

table23.to_csv(
    CELL_DIR / "stage16c_sparse_cell_moran_sensitivity.csv",
    index=False
)

print("\nStage 16C completed successfully.")
print("Original 47-cell dataset remains unchanged.")

STAGE 16C: SAMPLE-SIZE AUDIT
Original cells: 47
Retained cells: 45
Temporarily excluded cells: 2
Excluded Pixel_IDs: ['R015_C012', 'R022_C015']

Queen weights:
Remaining neighbour links: 66
Isolated cells: []

Rook weights:
Remaining neighbour links: 41
Isolated cells: []

TABLE 23: SPARSE-CELL SENSITIVITY
  Response Weights  Cells         Sample  Moran_I  Null_Mean  Null_SD  P_Two_Sided
Median_WLR   Queen     47      All cells  0.14442   -0.02206  0.10013       0.0965
Median_WLR   Queen     45 Exclude N < 30  0.15080   -0.02286  0.10535       0.1004
Median_WLR    Rook     47      All cells  0.20404   -0.02122  0.12673       0.0869
Median_WLR    Rook     45 Exclude N < 30  0.20207   -0.02340  0.13598       0.1060
  Mean_WLR   Queen     47      All cells  0.04986   -0.02179  0.10092       0.4027
  Mean_WLR   Queen     45 Exclude N < 30  0.04593   -0.02315  0.10571       0.4275
  Mean_WLR    Rook     47      All cells  0.04613   -0.02210  0.13006       0.4465
  Mean_WLR    Rook     45 Ex

In [8]:
# ============================================================
# STAGE 17A: LEAVE-ONE-CELL-OUT MORAN'S I INFLUENCE AUDIT
#
# Purpose:
# Identify cells that materially affect observed Moran's I.
#
# - Primary response: median WLR
# - Secondary response: mean WLR
# - Queen and Rook spatial weights
# - Each cell removed individually, then weights rebuilt
# - No permanent exclusions
# - No permutation p-values at this stage
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Align data and validate
# ------------------------------------------------------------

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

ids = np.asarray(cell_ids)

assert len(ids) == 47
assert len(set(ids)) == 47

responses = {
    "Median_WLR": pd.to_numeric(
        cells["WLR_Median"],
        errors="coerce"
    ).to_numpy(dtype=float),

    "Mean_WLR": pd.to_numeric(
        cells["WLR_Mean"],
        errors="coerce"
    ).to_numpy(dtype=float)
}

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

for values in responses.values():
    assert np.isfinite(values).all()

# ------------------------------------------------------------
# 2. Moran's I
# ------------------------------------------------------------

def morans_i_17(values, W):

    z = values - values.mean()

    if np.isclose(z @ z, 0):
        return np.nan

    if W.sum() == 0:
        return np.nan

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / (z @ z)
    )

# ------------------------------------------------------------
# 3. Rebuild weights after deleting one cell
# ------------------------------------------------------------

def rebuild_weights_17(W, keep):

    binary = (
        np.asarray(W) > 0
    ).astype(float)

    reduced = binary[np.ix_(keep, keep)]

    neighbour_counts = reduced.sum(axis=1)

    isolates = np.flatnonzero(
        neighbour_counts == 0
    )

    standardised = np.zeros_like(reduced)

    valid = neighbour_counts > 0

    standardised[valid] = (
        reduced[valid]
        / neighbour_counts[valid, None]
    )

    return standardised, isolates

# ------------------------------------------------------------
# 4. Calculate full and leave-one-out Moran's I
# ------------------------------------------------------------

records = []

for response_name, values in responses.items():

    for weights_name, W in weights.items():

        full_I = morans_i_17(values, W)

        for i, pixel_id in enumerate(ids):

            keep = np.ones(len(ids), dtype=bool)
            keep[i] = False

            reduced_W, isolates = rebuild_weights_17(
                W, keep
            )

            if len(isolates) > 0:

                # Do not silently apply a zero-neighbour convention.
                loo_I = np.nan

                isolate_ids = ";".join(
                    ids[keep][isolates]
                )

            else:

                loo_I = morans_i_17(
                    values[keep],
                    reduced_W
                )

                isolate_ids = ""

            records.append({
                "Response": response_name,
                "Weights": weights_name,
                "Removed_Pixel_ID": pixel_id,
                "Full_Moran_I": full_I,
                "Leave_One_Out_I": loo_I,
                "Delta_I": (
                    loo_I - full_I
                    if np.isfinite(loo_I)
                    else np.nan
                ),
                "Absolute_Delta_I": (
                    abs(loo_I - full_I)
                    if np.isfinite(loo_I)
                    else np.nan
                ),
                "Isolates_Created": len(isolates),
                "Isolate_IDs": isolate_ids
            })

influence = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. Table 24: Diagnostic coverage
# ------------------------------------------------------------

table24 = (
    influence
    .groupby(
        ["Response", "Weights"],
        as_index=False
    )
    .agg(
        Tests=("Removed_Pixel_ID", "size"),
        Valid_Tests=("Leave_One_Out_I", "count"),
        Tests_With_Isolates=(
            "Isolates_Created",
            lambda x: int((x > 0).sum())
        ),
        Maximum_Absolute_Delta_I=(
            "Absolute_Delta_I",
            "max"
        )
    )
)

print("\nTABLE 24: INFLUENCE DIAGNOSTIC COVERAGE")
print(table24.round(5).to_string(index=False))

# ------------------------------------------------------------
# 6. Table 25: Largest influences on median WLR
# ------------------------------------------------------------

table25 = (
    influence.loc[
        influence["Response"] == "Median_WLR"
    ]
    .sort_values(
        ["Weights", "Absolute_Delta_I"],
        ascending=[True, False]
    )
    .groupby("Weights", group_keys=False)
    .head(5)
)

print("\nTABLE 25: FIVE LARGEST INFLUENCES PER WEIGHTS")
print(
    table25[
        [
            "Weights",
            "Removed_Pixel_ID",
            "Full_Moran_I",
            "Leave_One_Out_I",
            "Delta_I",
            "Isolates_Created"
        ]
    ]
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Table 26: Previously identified cells
# ------------------------------------------------------------

focus_ids = [
    "R001_C005",
    "R024_C014",
    "R026_C018",
    "R015_C012",
    "R022_C015"
]

table26 = influence.loc[
    influence["Removed_Pixel_ID"].isin(focus_ids)
].copy()

print("\nTABLE 26: PREVIOUSLY IDENTIFIED CELLS")
print(
    table26[
        [
            "Response",
            "Weights",
            "Removed_Pixel_ID",
            "Leave_One_Out_I",
            "Delta_I",
            "Isolates_Created"
        ]
    ]
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

influence.to_csv(
    CELL_DIR / "stage17a_leave_one_cell_out_full.csv",
    index=False
)

table24.to_csv(
    CELL_DIR / "stage17a_diagnostic_coverage.csv",
    index=False
)

table25.to_csv(
    CELL_DIR / "stage17a_largest_median_influences.csv",
    index=False
)

table26.to_csv(
    CELL_DIR / "stage17a_focus_cells.csv",
    index=False
)

print("\nStage 17A completed.")
print("Original 47-cell dataset remains unchanged.")


TABLE 24: INFLUENCE DIAGNOSTIC COVERAGE
  Response Weights  Tests  Valid_Tests  Tests_With_Isolates  Maximum_Absolute_Delta_I
  Mean_WLR   Queen     47           46                    1                   0.05968
  Mean_WLR    Rook     47           41                    6                   0.06651
Median_WLR   Queen     47           46                    1                   0.07436
Median_WLR    Rook     47           41                    6                   0.09728

TABLE 25: FIVE LARGEST INFLUENCES PER WEIGHTS
Weights Removed_Pixel_ID  Full_Moran_I  Leave_One_Out_I  Delta_I  Isolates_Created
  Queen        R001_C003       0.14442          0.21878  0.07436                 0
  Queen        R020_C014       0.14442          0.08020 -0.06422                 0
  Queen        R024_C014       0.14442          0.09322 -0.05120                 0
  Queen        R023_C014       0.14442          0.09962 -0.04479                 0
  Queen        R024_C015       0.14442          0.17905  0.03463   

In [9]:
# ============================================================
# STAGE 17B: PERMUTATION SENSITIVITY FOR INFLUENTIAL CELLS
#
# Primary response: median WLR
# Weights: Queen and Rook
# Permutations: 9,999 unrestricted
#
# Each selected cell is removed individually.
# The original dataset is not modified.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare aligned inputs
# ------------------------------------------------------------

ids = np.asarray(cell_ids)

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
)

y = pd.to_numeric(
    cells["WLR_Median"],
    errors="coerce"
).to_numpy(dtype=float)

assert len(ids) == 47
assert np.isfinite(y).all()

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

focus_ids = [
    "R001_C003",
    "R020_C014",
    "R024_C014"
]

assert set(focus_ids).issubset(set(ids))

# ------------------------------------------------------------
# 2. Moran's I
# ------------------------------------------------------------

def morans_i_17b(values, W):

    z = values - values.mean()

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / (z @ z)
    )

# ------------------------------------------------------------
# 3. Rebuild weights after removing a cell
# ------------------------------------------------------------

def rebuild_weights_17b(W, keep):

    binary = (W > 0).astype(float)

    reduced = binary[np.ix_(keep, keep)]

    counts = reduced.sum(axis=1)

    isolates = np.flatnonzero(counts == 0)

    if len(isolates) > 0:
        return None, isolates

    return reduced / counts[:, None], isolates

# ------------------------------------------------------------
# 4. Permutation test
# ------------------------------------------------------------

def permutation_test_17b(
    values,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    observed = morans_i_17b(values, W)

    simulated = np.empty(permutations)

    for i in range(permutations):

        simulated[i] = morans_i_17b(
            rng.permutation(values),
            W
        )

    null_mean = simulated.mean()

    p_value = (
        1 + np.count_nonzero(
            np.abs(simulated - null_mean)
            >= abs(observed - null_mean)
        )
    ) / (permutations + 1)

    return {
        "Moran_I": observed,
        "Null_Mean": null_mean,
        "Null_SD": simulated.std(ddof=1),
        "P_Two_Sided": p_value
    }

# ------------------------------------------------------------
# 5. Full and leave-one-cell-out tests
# ------------------------------------------------------------

records = []

for weights_name, W in weights.items():

    full_result = permutation_test_17b(y, W)

    records.append({
        "Weights": weights_name,
        "Removed_Pixel_ID": "None",
        "Cells": 47,
        **full_result
    })

    for pixel_id in focus_ids:

        keep = ids != pixel_id

        reduced_W, isolates = rebuild_weights_17b(
            W, keep
        )

        if len(isolates) > 0:

            print(
                f"Skipping {weights_name}, {pixel_id}: "
                f"{len(isolates)} isolated cells."
            )

            continue

        result = permutation_test_17b(
            y[keep],
            reduced_W
        )

        records.append({
            "Weights": weights_name,
            "Removed_Pixel_ID": pixel_id,
            "Cells": int(keep.sum()),
            **result
        })

# ------------------------------------------------------------
# 6. Display and save Table 27
# ------------------------------------------------------------

table27 = pd.DataFrame(records)

print("\nTABLE 27: INFLUENTIAL-CELL PERMUTATION SENSITIVITY")

print(
    table27.round(5).to_string(index=False)
)

table27.to_csv(
    CELL_DIR / "stage17b_influential_cell_permutation.csv",
    index=False
)

print("\nStage 17B completed.")
print("Original 47-cell dataset remains unchanged.")


TABLE 27: INFLUENTIAL-CELL PERMUTATION SENSITIVITY
Weights Removed_Pixel_ID  Cells  Moran_I  Null_Mean  Null_SD  P_Two_Sided
  Queen             None     47  0.14442   -0.02206  0.10013       0.0965
  Queen        R001_C003     46  0.21878   -0.02299  0.09629       0.0189
  Queen        R020_C014     46  0.08020   -0.02294  0.10281       0.2991
  Queen        R024_C014     46  0.09322   -0.02168  0.09848       0.2252
   Rook             None     47  0.20404   -0.02122  0.12673       0.0869
   Rook        R001_C003     46  0.29903   -0.02181  0.12408       0.0242
   Rook        R020_C014     46  0.12492   -0.02312  0.13110       0.2287
   Rook        R024_C014     46  0.10676   -0.02155  0.12733       0.2488

Stage 17B completed.
Original 47-cell dataset remains unchanged.


In [10]:
# ============================================================
# STAGE 18A: LOCAL MORAN'S I - MEDIAN WLR
#
# Objectives:
# 1. Calculate local Moran's I for all 47 GLORYS cells.
# 2. Compare Queen and Rook spatial weights.
# 3. Run 9,999 unrestricted global permutations.
# 4. Calculate two-sided permutation p-values.
# 5. Apply Benjamini-Hochberg FDR correction separately
#    for Queen and Rook.
#
# No cells are excluded.
# This is an exploratory global-permutation local test.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate and align inputs
# ------------------------------------------------------------

ids = np.asarray(cell_ids)

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

y = pd.to_numeric(
    cells["WLR_Median"],
    errors="coerce"
).to_numpy(dtype=float)

assert len(ids) == 47
assert len(set(ids)) == 47
assert np.isfinite(y).all()
assert np.var(y) > 0

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

for name, W in weights.items():
    assert W.shape == (47, 47), name
    assert np.allclose(W.sum(axis=1), 1), name
    assert np.allclose(np.diag(W), 0), name

# ------------------------------------------------------------
# 2. Benjamini-Hochberg FDR correction
# ------------------------------------------------------------

def bh_adjust(p_values):

    p = np.asarray(p_values, dtype=float)

    n = len(p)

    order = np.argsort(p)

    ranked = p[order]

    adjusted_ranked = (
        ranked * n / np.arange(1, n + 1)
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0,
        1
    )

    adjusted = np.empty(n, dtype=float)

    adjusted[order] = adjusted_ranked

    return adjusted

# ------------------------------------------------------------
# 3. Local Moran's I and permutation inference
# ------------------------------------------------------------

def local_moran_test(
    values,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    z = values - values.mean()

    m2 = np.mean(z ** 2)

    # Observed spatial lag and local Moran's I.
    lag = W @ z

    observed = z * lag / m2

    # Unrestricted global permutations:
    # each iteration permutes the full response vector.
    simulated = np.empty(
        (permutations, len(values)),
        dtype=float
    )

    for i in range(permutations):

        z_perm = rng.permutation(z)

        lag_perm = W @ z_perm

        simulated[i] = (
            z_perm * lag_perm / m2
        )

    null_mean = simulated.mean(axis=0)

    null_sd = simulated.std(
        axis=0,
        ddof=1
    )

    # Two-sided test centred on each cell's simulated null.
    deviations = np.abs(
        simulated - null_mean
    )

    observed_deviation = np.abs(
        observed - null_mean
    )

    p_values = (
        1 + np.sum(
            deviations >= observed_deviation,
            axis=0
        )
    ) / (permutations + 1)

    p_fdr = bh_adjust(p_values)

    # Quadrants are based on observed centred values.
    quadrants = np.where(
        (z >= 0) & (lag >= 0),
        "HH",
        np.where(
            (z < 0) & (lag < 0),
            "LL",
            np.where(
                (z >= 0) & (lag < 0),
                "HL",
                "LH"
            )
        )
    )

    # Significance categories are explicitly separate
    # from the observed HH/LL/HL/LH quadrants.
    fdr_category = np.where(
        p_fdr < 0.05,
        quadrants,
        "Not significant"
    )

    return pd.DataFrame({
        "Pixel_ID": ids,
        "WLR_Median": values,
        "Centered_WLR": z,
        "Spatial_Lag": lag,
        "Local_Moran_I": observed,
        "Null_Mean": null_mean,
        "Null_SD": null_sd,
        "P_Raw": p_values,
        "P_FDR": p_fdr,
        "Quadrant": quadrants,
        "FDR_Category": fdr_category
    })

# ------------------------------------------------------------
# 4. Run Queen and Rook analyses
# ------------------------------------------------------------

results = []

for name, W in weights.items():

    result = local_moran_test(
        values=y,
        W=W,
        permutations=9999,
        seed=42
    )

    result.insert(1, "Weights", name)

    results.append(result)

local_results = pd.concat(
    results,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. Table 28: Significance summary
# ------------------------------------------------------------

summary_records = []

for name in weights:

    subset = local_results.loc[
        local_results["Weights"] == name
    ]

    summary_records.append({
        "Weights": name,
        "Cells": len(subset),
        "Raw_P_Less_005": int(
            (subset["P_Raw"] < 0.05).sum()
        ),
        "FDR_P_Less_005": int(
            (subset["P_FDR"] < 0.05).sum()
        ),
        "FDR_HH": int(
            (subset["FDR_Category"] == "HH").sum()
        ),
        "FDR_LL": int(
            (subset["FDR_Category"] == "LL").sum()
        ),
        "FDR_HL": int(
            (subset["FDR_Category"] == "HL").sum()
        ),
        "FDR_LH": int(
            (subset["FDR_Category"] == "LH").sum()
        )
    })

table28 = pd.DataFrame(summary_records)

print("\nTABLE 28: LOCAL MORAN SIGNIFICANCE SUMMARY")

print(
    table28.to_string(index=False)
)

# ------------------------------------------------------------
# 6. Table 29: Cells significant after FDR
# ------------------------------------------------------------

table29 = local_results.loc[
    local_results["P_FDR"] < 0.05,
    [
        "Weights",
        "Pixel_ID",
        "WLR_Median",
        "Spatial_Lag",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant"
    ]
].sort_values(
    ["Weights", "P_FDR", "Pixel_ID"]
)

print("\nTABLE 29: FDR-SIGNIFICANT LOCAL CELLS")

if table29.empty:
    print("No cells significant after FDR correction.")
else:
    print(
        table29.round(5).to_string(index=False)
    )

# ------------------------------------------------------------
# 7. Save all outputs
# ------------------------------------------------------------

local_results.to_csv(
    CELL_DIR / "stage18a_local_moran_all_cells.csv",
    index=False
)

table28.to_csv(
    CELL_DIR / "stage18a_local_moran_summary.csv",
    index=False
)

table29.to_csv(
    CELL_DIR / "stage18a_local_moran_fdr_significant.csv",
    index=False
)

print("\nStage 18A completed successfully.")
print("All 47 cells retained.")
print("FDR correction applied separately to Queen and Rook.")


TABLE 28: LOCAL MORAN SIGNIFICANCE SUMMARY
Weights  Cells  Raw_P_Less_005  FDR_P_Less_005  FDR_HH  FDR_LL  FDR_HL  FDR_LH
  Queen     47               5               0       0       0       0       0
   Rook     47               5               0       0       0       0       0

TABLE 29: FDR-SIGNIFICANT LOCAL CELLS
No cells significant after FDR correction.

Stage 18A completed successfully.
All 47 cells retained.
FDR correction applied separately to Queen and Rook.


In [11]:
# ============================================================
# STAGE 18B: CONDITIONAL LOCAL MORAN'S I
#
# Primary response: median WLR
# Weights: Queen and Rook
# Permutations: 9,999 per focal cell
#
# Conditional randomisation:
# - Hold the focal cell's value fixed.
# - Permute values among all other cells.
# - Recalculate the focal cell's spatial lag.
#
# FDR correction is applied separately for each weights matrix.
# All 47 cells retained.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Align and validate inputs
# ------------------------------------------------------------

ids = np.asarray(cell_ids)

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
)

y = pd.to_numeric(
    cells["WLR_Median"],
    errors="coerce"
).to_numpy(dtype=float)

assert len(ids) == 47
assert np.isfinite(y).all()
assert np.var(y) > 0

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

for name, W in weights.items():
    assert W.shape == (47, 47), name
    assert np.allclose(W.sum(axis=1), 1), name
    assert np.allclose(np.diag(W), 0), name

# ------------------------------------------------------------
# 2. Benjamini-Hochberg FDR correction
# ------------------------------------------------------------

def bh_adjust_18b(p_values):

    p = np.asarray(p_values, dtype=float)

    n = len(p)
    order = np.argsort(p)
    ranked = p[order]

    adjusted_ranked = (
        ranked * n / np.arange(1, n + 1)
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted = np.empty(n, dtype=float)

    adjusted[order] = np.clip(
        adjusted_ranked,
        0,
        1
    )

    return adjusted

# ------------------------------------------------------------
# 3. Conditional local permutation test
# ------------------------------------------------------------

def conditional_local_moran(
    values,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    n = len(values)

    z = values - values.mean()
    m2 = np.mean(z ** 2)

    observed_lag = W @ z
    observed_I = z * observed_lag / m2

    null_mean = np.empty(n)
    null_sd = np.empty(n)
    p_raw = np.empty(n)

    for focal in range(n):

        # All cells except the focal cell.
        other_indices = np.delete(
            np.arange(n),
            focal
        )

        other_values = z[other_indices]

        # Weights from the focal cell to all other cells.
        focal_weights = W[
            focal,
            other_indices
        ]

        simulated = np.empty(permutations)

        for iteration in range(permutations):

            shuffled = rng.permutation(
                other_values
            )

            simulated_lag = (
                focal_weights @ shuffled
            )

            simulated[iteration] = (
                z[focal] * simulated_lag / m2
            )

        null_mean[focal] = simulated.mean()

        null_sd[focal] = simulated.std(ddof=1)

        # Two-sided permutation p-value centred on
        # the focal cell's simulated null mean.
        p_raw[focal] = (
            1 + np.count_nonzero(
                np.abs(
                    simulated - null_mean[focal]
                )
                >= abs(
                    observed_I[focal]
                    - null_mean[focal]
                )
            )
        ) / (permutations + 1)

    p_fdr = bh_adjust_18b(p_raw)

    quadrants = np.where(
        (z >= 0) & (observed_lag >= 0),
        "HH",
        np.where(
            (z < 0) & (observed_lag < 0),
            "LL",
            np.where(
                (z >= 0) & (observed_lag < 0),
                "HL",
                "LH"
            )
        )
    )

    fdr_category = np.where(
        p_fdr < 0.05,
        quadrants,
        "Not significant"
    )

    return pd.DataFrame({
        "Pixel_ID": ids,
        "WLR_Median": values,
        "Spatial_Lag": observed_lag,
        "Local_Moran_I": observed_I,
        "Null_Mean": null_mean,
        "Null_SD": null_sd,
        "P_Raw": p_raw,
        "P_FDR": p_fdr,
        "Quadrant": quadrants,
        "FDR_Category": fdr_category
    })

# ------------------------------------------------------------
# 4. Run Queen and Rook
# ------------------------------------------------------------

results = []

for name, W in weights.items():

    result = conditional_local_moran(
        values=y,
        W=W,
        permutations=9999,
        seed=42
    )

    result.insert(1, "Weights", name)

    results.append(result)

conditional_results = pd.concat(
    results,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. Table 30: Conditional test summary
# ------------------------------------------------------------

summary = []

for name in weights:

    subset = conditional_results.loc[
        conditional_results["Weights"] == name
    ]

    summary.append({
        "Weights": name,
        "Cells": len(subset),
        "Raw_P_Less_005": int(
            (subset["P_Raw"] < 0.05).sum()
        ),
        "FDR_P_Less_005": int(
            (subset["P_FDR"] < 0.05).sum()
        ),
        "FDR_HH": int(
            (subset["FDR_Category"] == "HH").sum()
        ),
        "FDR_LL": int(
            (subset["FDR_Category"] == "LL").sum()
        ),
        "FDR_HL": int(
            (subset["FDR_Category"] == "HL").sum()
        ),
        "FDR_LH": int(
            (subset["FDR_Category"] == "LH").sum()
        )
    })

table30 = pd.DataFrame(summary)

print("\nTABLE 30: CONDITIONAL LOCAL MORAN SUMMARY")
print(table30.to_string(index=False))

# ------------------------------------------------------------
# 6. Table 31: FDR-significant cells
# ------------------------------------------------------------

table31 = conditional_results.loc[
    conditional_results["P_FDR"] < 0.05,
    [
        "Weights",
        "Pixel_ID",
        "WLR_Median",
        "Spatial_Lag",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant"
    ]
].sort_values(
    ["Weights", "P_FDR", "Pixel_ID"]
)

print("\nTABLE 31: FDR-SIGNIFICANT CONDITIONAL CELLS")

if table31.empty:
    print("No cells significant after FDR correction.")
else:
    print(
        table31.round(5).to_string(index=False)
    )

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

conditional_results.to_csv(
    CELL_DIR / "stage18b_conditional_local_moran_all_cells.csv",
    index=False
)

table30.to_csv(
    CELL_DIR / "stage18b_conditional_local_moran_summary.csv",
    index=False
)

table31.to_csv(
    CELL_DIR / "stage18b_conditional_local_moran_fdr_significant.csv",
    index=False
)

print("\nStage 18B completed successfully.")
print("All 47 cells retained.")
print("Conditional permutations and FDR correction applied.")


TABLE 30: CONDITIONAL LOCAL MORAN SUMMARY
Weights  Cells  Raw_P_Less_005  FDR_P_Less_005  FDR_HH  FDR_LL  FDR_HL  FDR_LH
  Queen     47               3               0       0       0       0       0
   Rook     47               2               0       0       0       0       0

TABLE 31: FDR-SIGNIFICANT CONDITIONAL CELLS
No cells significant after FDR correction.

Stage 18B completed successfully.
All 47 cells retained.
Conditional permutations and FDR correction applied.


In [12]:
# ============================================================
# STAGE 19: LOCAL MORAN CONSISTENCY AUDIT
#
# Compare:
# - Stage 18A: unrestricted global permutations
# - Stage 18B: conditional local permutations
# - Queen and Rook weights
#
# No new statistical tests.
# No cells excluded.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Verify previous outputs
# ------------------------------------------------------------

required = [
    "local_results",
    "conditional_results",
    "cell_ids",
    "CELL_DIR"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing objects: " + ", ".join(missing)
    )

assert len(local_results) == 94
assert len(conditional_results) == 94

# ------------------------------------------------------------
# 2. Standardise the two result tables
# ------------------------------------------------------------

global_test = local_results[
    [
        "Pixel_ID",
        "Weights",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant"
    ]
].copy()

global_test["Method"] = "Global permutation"

conditional_test = conditional_results[
    [
        "Pixel_ID",
        "Weights",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant"
    ]
].copy()

conditional_test["Method"] = "Conditional permutation"

combined = pd.concat(
    [global_test, conditional_test],
    ignore_index=True
)

assert len(combined) == 188

assert not combined.duplicated(
    ["Pixel_ID", "Weights", "Method"]
).any()

assert set(combined["Pixel_ID"]) == set(cell_ids)

# ------------------------------------------------------------
# 3. Add descriptive significance flags
# ------------------------------------------------------------

combined["Raw_Significant"] = (
    combined["P_Raw"] < 0.05
)

combined["FDR_Significant"] = (
    combined["P_FDR"] < 0.05
)

# ------------------------------------------------------------
# 4. Table 32: Method comparison
# ------------------------------------------------------------

table32 = (
    combined
    .groupby(
        ["Method", "Weights"],
        as_index=False
    )
    .agg(
        Cells=("Pixel_ID", "size"),
        Raw_Significant=("Raw_Significant", "sum"),
        FDR_Significant=("FDR_Significant", "sum"),
        Minimum_Raw_P=("P_Raw", "min"),
        Minimum_FDR_P=("P_FDR", "min")
    )
)

print("\nTABLE 32: LOCAL MORAN METHOD COMPARISON")

print(
    table32.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 33: All nominally significant results
# ------------------------------------------------------------

table33 = (
    combined.loc[
        combined["Raw_Significant"]
    ]
    .sort_values(
        ["Pixel_ID", "Weights", "Method"]
    )
)

print("\nTABLE 33: NOMINALLY SIGNIFICANT RESULTS")

if table33.empty:
    print("No nominally significant cells.")
else:
    print(
        table33[
            [
                "Pixel_ID",
                "Weights",
                "Method",
                "Quadrant",
                "Local_Moran_I",
                "P_Raw",
                "P_FDR"
            ]
        ]
        .round(5)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 6. Table 34: Agreement across the four specifications
# ------------------------------------------------------------

agreement = (
    combined
    .groupby("Pixel_ID")
    .agg(
        Raw_Significant_Tests=(
            "Raw_Significant",
            "sum"
        ),
        FDR_Significant_Tests=(
            "FDR_Significant",
            "sum"
        ),
        Minimum_Raw_P=(
            "P_Raw",
            "min"
        ),
        Minimum_FDR_P=(
            "P_FDR",
            "min"
        ),
        Quadrant_Count=(
            "Quadrant",
            "nunique"
        )
    )
    .reset_index()
)

agreement["Quadrant_Agreement"] = np.where(
    agreement["Quadrant_Count"] == 1,
    "Same across all four",
    "Differs across weights/methods"
)

agreement = agreement.sort_values(
    [
        "Raw_Significant_Tests",
        "Minimum_Raw_P"
    ],
    ascending=[False, True]
)

table34 = agreement.loc[
    agreement["Raw_Significant_Tests"] > 0
].copy()

print("\nTABLE 34: CELLS WITH AT LEAST ONE NOMINAL RESULT")

if table34.empty:
    print("No cells to report.")
else:
    print(
        table34[
            [
                "Pixel_ID",
                "Raw_Significant_Tests",
                "FDR_Significant_Tests",
                "Minimum_Raw_P",
                "Minimum_FDR_P",
                "Quadrant_Agreement"
            ]
        ]
        .round(5)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 7. Verify that FDR results match Stages 18A and 18B
# ------------------------------------------------------------

assert combined["FDR_Significant"].sum() == 0, (
    "Unexpected FDR-significant result. "
    "Review the earlier stage outputs."
)

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

combined.to_csv(
    CELL_DIR / "stage19_local_moran_method_comparison_full.csv",
    index=False
)

table32.to_csv(
    CELL_DIR / "stage19_method_summary.csv",
    index=False
)

table33.to_csv(
    CELL_DIR / "stage19_nominal_results.csv",
    index=False
)

table34.to_csv(
    CELL_DIR / "stage19_cell_agreement.csv",
    index=False
)

print("\nStage 19 completed successfully.")
print("All 47 cells retained.")
print("No new permutation tests performed.")


TABLE 32: LOCAL MORAN METHOD COMPARISON
                 Method Weights  Cells  Raw_Significant  FDR_Significant  Minimum_Raw_P  Minimum_FDR_P
Conditional permutation   Queen     47                3                0         0.0120        0.50055
Conditional permutation    Rook     47                2                0         0.0097        0.28200
     Global permutation   Queen     47                5                0         0.0145        0.35532
     Global permutation    Rook     47                5                0         0.0145        0.20680

TABLE 33: NOMINALLY SIGNIFICANT RESULTS
 Pixel_ID Weights                  Method Quadrant  Local_Moran_I  P_Raw   P_FDR
R001_C003   Queen      Global permutation       LH       -1.61230 0.0343 0.35532
R001_C003    Rook      Global permutation       LH       -1.61230 0.0343 0.32242
R018_C014   Queen Conditional permutation       HH        0.21601 0.0120 0.50055
R018_C014    Rook Conditional permutation       HH        0.30259 0.0120 0.2820

In [13]:
# ============================================================
# STAGE 20: EXPORT LOCAL MORAN RESULTS TO GIS
#
# Outputs:
# 1. GeoPackage with Queen and Rook results
# 2. CSV with all local Moran attributes
# 3. GIS export validation summary
#
# Primary inference: conditional local permutations
# Global permutations retained as sensitivity results
#
# All 47 original GLORYS cell geometries retained.
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------
# 1. Validate source objects
# ------------------------------------------------------------

assert len(analysis_grid) == 47
assert len(local_results) == 94
assert len(conditional_results) == 94

assert analysis_grid["Pixel_ID"].is_unique

assert analysis_grid.crs is not None, (
    "Missing CRS. Check the original GLORYS grid."
)

# ------------------------------------------------------------
# 2. Prepare original polygon geometry
# ------------------------------------------------------------

geometry = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .reset_index()
    .copy()
)

assert geometry.geometry.notna().all()

assert geometry.geometry.is_valid.all()

assert geometry["Pixel_ID"].is_unique

# ------------------------------------------------------------
# 3. Prepare conditional results
# ------------------------------------------------------------

conditional = conditional_results[
    [
        "Pixel_ID",
        "Weights",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant",
        "FDR_Category"
    ]
].copy()

conditional = conditional.rename(
    columns={
        "Local_Moran_I": "Local_I_Conditional",
        "P_Raw": "P_Conditional",
        "P_FDR": "FDR_Conditional",
        "Quadrant": "Quadrant_Conditional",
        "FDR_Category": "FDR_Category_Conditional"
    }
)

# ------------------------------------------------------------
# 4. Prepare global-permutation results
# ------------------------------------------------------------

global_perm = local_results[
    [
        "Pixel_ID",
        "Weights",
        "Local_Moran_I",
        "P_Raw",
        "P_FDR",
        "Quadrant"
    ]
].copy()

global_perm = global_perm.rename(
    columns={
        "Local_Moran_I": "Local_I_Global",
        "P_Raw": "P_Global",
        "P_FDR": "FDR_Global",
        "Quadrant": "Quadrant_Global"
    }
)

# ------------------------------------------------------------
# 5. Merge results
# ------------------------------------------------------------

statistics = conditional.merge(
    global_perm,
    on=["Pixel_ID", "Weights"],
    how="inner",
    validate="one_to_one"
)

assert len(statistics) == 94

# Observed local Moran's I and quadrants must agree
# because only the permutation procedures differ.

assert np.allclose(
    statistics["Local_I_Conditional"],
    statistics["Local_I_Global"]
)

assert (
    statistics["Quadrant_Conditional"]
    == statistics["Quadrant_Global"]
).all()

# ------------------------------------------------------------
# 6. Add transparent GIS classification fields
# ------------------------------------------------------------

statistics["Nominal_Conditional"] = np.where(
    statistics["P_Conditional"] < 0.05,
    "Nominal p < 0.05",
    "Not nominally significant"
)

statistics["Nominal_Global"] = np.where(
    statistics["P_Global"] < 0.05,
    "Nominal p < 0.05",
    "Not nominally significant"
)

statistics["FDR_Confirmed"] = np.where(
    statistics["FDR_Conditional"] < 0.05,
    statistics["Quadrant_Conditional"],
    "Not significant"
)

# ------------------------------------------------------------
# 7. Join statistics to original GLORYS polygons
# ------------------------------------------------------------

spatial = geometry.merge(
    statistics,
    on="Pixel_ID",
    how="inner",
    validate="one_to_many"
)

spatial = gpd.GeoDataFrame(
    spatial,
    geometry="geometry",
    crs=analysis_grid.crs
)

assert len(spatial) == 94

assert spatial.groupby("Weights").size().to_dict() == {
    "Queen": 47,
    "Rook": 47
}

assert spatial.geometry.notna().all()

assert spatial.geometry.is_valid.all()

# ------------------------------------------------------------
# 8. Separate Queen and Rook layers
# ------------------------------------------------------------

queen_gis = spatial.loc[
    spatial["Weights"] == "Queen"
].copy()

rook_gis = spatial.loc[
    spatial["Weights"] == "Rook"
].copy()

assert len(queen_gis) == 47
assert len(rook_gis) == 47

# ------------------------------------------------------------
# 9. Export GeoPackage
# ------------------------------------------------------------

gpkg_path = (
    CELL_DIR
    / "stage20_local_moran_GIS.gpkg"
)

# Remove only the previous Stage 20 export if present.
# This does not touch the verified source GeoPackage.

if gpkg_path.exists():
    gpkg_path.unlink()

queen_gis.to_file(
    gpkg_path,
    layer="local_moran_queen",
    driver="GPKG"
)

rook_gis.to_file(
    gpkg_path,
    layer="local_moran_rook",
    driver="GPKG",
    mode="a"
)

# ------------------------------------------------------------
# 10. Export full attribute table
# ------------------------------------------------------------

csv_path = (
    CELL_DIR
    / "stage20_local_moran_GIS_attributes.csv"
)

spatial.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# ------------------------------------------------------------
# 11. Validate written GeoPackage
# ------------------------------------------------------------

queen_check = gpd.read_file(
    gpkg_path,
    layer="local_moran_queen"
)

rook_check = gpd.read_file(
    gpkg_path,
    layer="local_moran_rook"
)

assert len(queen_check) == 47
assert len(rook_check) == 47

assert queen_check.crs == analysis_grid.crs
assert rook_check.crs == analysis_grid.crs

assert set(queen_check["Pixel_ID"]) == set(cell_ids)
assert set(rook_check["Pixel_ID"]) == set(cell_ids)

assert (
    queen_check["FDR_Confirmed"]
    == "Not significant"
).all()

assert (
    rook_check["FDR_Confirmed"]
    == "Not significant"
).all()

# ------------------------------------------------------------
# 12. Table 35: Export validation
# ------------------------------------------------------------

table35 = pd.DataFrame([
    {
        "Layer": "local_moran_queen",
        "Cells": len(queen_check),
        "Unique_Pixel_IDs": queen_check["Pixel_ID"].nunique(),
        "Valid_Geometries": int(
            queen_check.geometry.is_valid.sum()
        ),
        "FDR_Significant": int(
            (
                queen_check["FDR_Confirmed"]
                != "Not significant"
            ).sum()
        )
    },
    {
        "Layer": "local_moran_rook",
        "Cells": len(rook_check),
        "Unique_Pixel_IDs": rook_check["Pixel_ID"].nunique(),
        "Valid_Geometries": int(
            rook_check.geometry.is_valid.sum()
        ),
        "FDR_Significant": int(
            (
                rook_check["FDR_Confirmed"]
                != "Not significant"
            ).sum()
        )
    }
])

print("\nTABLE 35: GIS EXPORT VALIDATION")
print(table35.to_string(index=False))

print("\nGeoPackage:", gpkg_path)
print("CSV:", csv_path)

print("\nStage 20 completed successfully.")
print("Original 47-cell dataset remains unchanged.")


TABLE 35: GIS EXPORT VALIDATION
            Layer  Cells  Unique_Pixel_IDs  Valid_Geometries  FDR_Significant
local_moran_queen     47                47                47                0
 local_moran_rook     47                47                47                0

GeoPackage: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\stage20_local_moran_GIS.gpkg
CSV: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\Geostatistics\GLORYS_Cell_Level\stage20_local_moran_GIS_attributes.csv

Stage 20 completed successfully.
Original 47-cell dataset remains unchanged.


In [14]:
# ============================================================
# STAGE 21A: CELL-LEVEL VARIABLE AND ASSOCIATION AUDIT
#
# Purpose:
# 1. Validate the 47-cell analytical dataset.
# 2. Check missing and non-finite values.
# 3. Summarise shoreline and sea-level variables.
# 4. Calculate descriptive Pearson and Spearman correlations.
# 5. Identify extreme observations for inspection.
#
# No regression models.
# No statistical significance tests.
# No cells excluded.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare and align the original 47-cell dataset
# ------------------------------------------------------------

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

assert len(cells) == 47
assert cells.index.is_unique

variables = {
    "WLR_Median": "Median shoreline change (m/yr)",
    "WLR_Mean": "Mean shoreline change (m/yr)",
    "SL_Trend_mm_yr": "Sea-level trend (mm/yr)",
    "SL_Variability_m": "Sea-level variability (m)",
    "Valid_WLR_Count": "Valid transects per cell"
}

data = cells[list(variables)].apply(
    pd.to_numeric,
    errors="coerce"
)

data.index.name = "Pixel_ID"

# ------------------------------------------------------------
# 2. Table 36: Completeness audit
# ------------------------------------------------------------

completeness = []

for column, description in variables.items():

    values = data[column].to_numpy(dtype=float)

    completeness.append({
        "Variable": column,
        "Description": description,
        "Cells": len(values),
        "Missing": int(np.isnan(values).sum()),
        "Infinite": int(np.isinf(values).sum()),
        "Finite": int(np.isfinite(values).sum())
    })

table36 = pd.DataFrame(completeness)

print("\nTABLE 36: VARIABLE COMPLETENESS")
print(table36.to_string(index=False))

# Do not silently proceed with incomplete data.
if not np.isfinite(data.to_numpy(dtype=float)).all():

    raise RuntimeError(
        "Missing or non-finite values detected. "
        "Review Table 36 before proceeding."
    )

assert int(data["Valid_WLR_Count"].sum()) == 12452

# ------------------------------------------------------------
# 3. Table 37: Descriptive statistics
# ------------------------------------------------------------

table37 = (
    data
    .describe()
    .T
    .rename(
        columns={
            "count": "N",
            "mean": "Mean",
            "std": "SD",
            "min": "Minimum",
            "25%": "Q1",
            "50%": "Median",
            "75%": "Q3",
            "max": "Maximum"
        }
    )
    [
        [
            "N",
            "Mean",
            "SD",
            "Minimum",
            "Q1",
            "Median",
            "Q3",
            "Maximum"
        ]
    ]
)

print("\nTABLE 37: DESCRIPTIVE STATISTICS")

print(
    table37.round(5).to_string()
)

# ------------------------------------------------------------
# 4. Table 38: Descriptive correlation matrix
# ------------------------------------------------------------

correlation_variables = [
    "WLR_Median",
    "WLR_Mean",
    "SL_Trend_mm_yr",
    "SL_Variability_m"
]

pearson = data[
    correlation_variables
].corr(method="pearson")

spearman = data[
    correlation_variables
].corr(method="spearman")

table38 = pd.DataFrame([
    {
        "Shoreline_Response": response,
        "Sea_Level_Predictor": predictor,
        "Pearson_r": pearson.loc[
            response, predictor
        ],
        "Spearman_rho": spearman.loc[
            response, predictor
        ]
    }

    for response in [
        "WLR_Median",
        "WLR_Mean"
    ]

    for predictor in [
        "SL_Trend_mm_yr",
        "SL_Variability_m"
    ]
])

print("\nTABLE 38: DESCRIPTIVE CORRELATIONS")

print(
    table38.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 39: Predictor-to-predictor relationship
# ------------------------------------------------------------

table39 = pd.DataFrame([
    {
        "Predictor_1": "SL_Trend_mm_yr",
        "Predictor_2": "SL_Variability_m",
        "Pearson_r": pearson.loc[
            "SL_Trend_mm_yr",
            "SL_Variability_m"
        ],
        "Spearman_rho": spearman.loc[
            "SL_Trend_mm_yr",
            "SL_Variability_m"
        ]
    }
])

print("\nTABLE 39: SEA-LEVEL PREDICTOR CORRELATION")

print(
    table39.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Table 40: Extreme values for inspection
# ------------------------------------------------------------

extreme_records = []

for column in correlation_variables:

    smallest = data[column].nsmallest(3)

    largest = data[column].nlargest(3)

    for pixel_id, value in smallest.items():

        extreme_records.append({
            "Variable": column,
            "Extreme": "Lowest",
            "Pixel_ID": pixel_id,
            "Value": value,
            "Valid_WLR_Count": int(
                data.loc[pixel_id, "Valid_WLR_Count"]
            )
        })

    for pixel_id, value in largest.items():

        extreme_records.append({
            "Variable": column,
            "Extreme": "Highest",
            "Pixel_ID": pixel_id,
            "Value": value,
            "Valid_WLR_Count": int(
                data.loc[pixel_id, "Valid_WLR_Count"]
            )
        })

table40 = pd.DataFrame(extreme_records)

print("\nTABLE 40: EXTREME VALUES FOR INSPECTION")

print(
    table40.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

table36.to_csv(
    CELL_DIR / "stage21a_variable_completeness.csv",
    index=False
)

table37.to_csv(
    CELL_DIR / "stage21a_descriptive_statistics.csv"
)

table38.to_csv(
    CELL_DIR / "stage21a_descriptive_correlations.csv",
    index=False
)

table39.to_csv(
    CELL_DIR / "stage21a_predictor_correlation.csv",
    index=False
)

table40.to_csv(
    CELL_DIR / "stage21a_extreme_values.csv",
    index=False
)

print("\nStage 21A completed successfully.")
print("All 47 cells retained.")
print("No regression or significance tests performed.")


TABLE 36: VARIABLE COMPLETENESS
        Variable                    Description  Cells  Missing  Infinite  Finite
      WLR_Median Median shoreline change (m/yr)     47        0         0      47
        WLR_Mean   Mean shoreline change (m/yr)     47        0         0      47
  SL_Trend_mm_yr        Sea-level trend (mm/yr)     47        0         0      47
SL_Variability_m      Sea-level variability (m)     47        0         0      47
 Valid_WLR_Count       Valid transects per cell     47        0         0      47

TABLE 37: DESCRIPTIVE STATISTICS
                     N       Mean         SD  Minimum         Q1     Median         Q3    Maximum
WLR_Median        47.0    0.01779    0.92117 -2.13469   -0.17959    0.02577    0.19510    4.38305
WLR_Mean          47.0    0.25820    2.06294 -5.86627   -0.22110    0.03060    0.35287    9.69655
SL_Trend_mm_yr    47.0    3.93307    0.04620  3.82745    3.90482    3.93592    3.96422    4.01456
SL_Variability_m  47.0    0.02724    0.00041  0.0

In [15]:
# ============================================================
# STAGE 21B: CORRELATION INFLUENCE AUDIT
#
# Purpose:
# 1. Examine leave-one-cell-out correlations.
# 2. Identify influential cells.
# 3. Assess stability of descriptive associations.
#
# Primary response: median WLR
# Predictors: sea-level trend and variability
#
# No regression.
# No significance tests.
# No permanent exclusions.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare aligned data
# ------------------------------------------------------------

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

columns = [
    "WLR_Median",
    "SL_Trend_mm_yr",
    "SL_Variability_m",
    "Valid_WLR_Count"
]

data = cells[columns].apply(
    pd.to_numeric,
    errors="coerce"
)

assert len(data) == 47
assert np.isfinite(data.to_numpy(dtype=float)).all()

# ------------------------------------------------------------
# 2. Calculate full-sample correlations
# ------------------------------------------------------------

predictors = [
    "SL_Trend_mm_yr",
    "SL_Variability_m"
]

records = []

for predictor in predictors:

    full_pearson = data[
        "WLR_Median"
    ].corr(
        data[predictor],
        method="pearson"
    )

    full_spearman = data[
        "WLR_Median"
    ].corr(
        data[predictor],
        method="spearman"
    )

    # --------------------------------------------------------
    # 3. Leave one cell out at a time
    # --------------------------------------------------------

    for pixel_id in data.index:

        subset = data.drop(
            index=pixel_id
        )

        loo_pearson = subset[
            "WLR_Median"
        ].corr(
            subset[predictor],
            method="pearson"
        )

        loo_spearman = subset[
            "WLR_Median"
        ].corr(
            subset[predictor],
            method="spearman"
        )

        records.append({
            "Predictor": predictor,
            "Removed_Pixel_ID": pixel_id,
            "Valid_WLR_Count": int(
                data.loc[
                    pixel_id,
                    "Valid_WLR_Count"
                ]
            ),
            "Full_Pearson": full_pearson,
            "LOO_Pearson": loo_pearson,
            "Delta_Pearson": (
                loo_pearson - full_pearson
            ),
            "Full_Spearman": full_spearman,
            "LOO_Spearman": loo_spearman,
            "Delta_Spearman": (
                loo_spearman - full_spearman
            )
        })

influence21b = pd.DataFrame(records)

assert len(influence21b) == 94

# ------------------------------------------------------------
# 4. Table 41: Correlation stability
# ------------------------------------------------------------

table41 = (
    influence21b
    .groupby(
        "Predictor",
        as_index=False
    )
    .agg(
        Full_Pearson=("Full_Pearson", "first"),
        Min_LOO_Pearson=("LOO_Pearson", "min"),
        Max_LOO_Pearson=("LOO_Pearson", "max"),
        Full_Spearman=("Full_Spearman", "first"),
        Min_LOO_Spearman=("LOO_Spearman", "min"),
        Max_LOO_Spearman=("LOO_Spearman", "max")
    )
)

print("\nTABLE 41: CORRELATION STABILITY")

print(
    table41.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 42: Largest Pearson influences
# ------------------------------------------------------------

influence21b["Abs_Delta_Pearson"] = (
    influence21b["Delta_Pearson"].abs()
)

table42 = (
    influence21b
    .sort_values(
        [
            "Predictor",
            "Abs_Delta_Pearson"
        ],
        ascending=[True, False]
    )
    .groupby(
        "Predictor",
        group_keys=False
    )
    .head(5)
)

print("\nTABLE 42: LARGEST PEARSON INFLUENCES")

print(
    table42[
        [
            "Predictor",
            "Removed_Pixel_ID",
            "Valid_WLR_Count",
            "Full_Pearson",
            "LOO_Pearson",
            "Delta_Pearson"
        ]
    ]
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Table 43: Largest Spearman influences
# ------------------------------------------------------------

influence21b["Abs_Delta_Spearman"] = (
    influence21b["Delta_Spearman"].abs()
)

table43 = (
    influence21b
    .sort_values(
        [
            "Predictor",
            "Abs_Delta_Spearman"
        ],
        ascending=[True, False]
    )
    .groupby(
        "Predictor",
        group_keys=False
    )
    .head(5)
)

print("\nTABLE 43: LARGEST SPEARMAN INFLUENCES")

print(
    table43[
        [
            "Predictor",
            "Removed_Pixel_ID",
            "Valid_WLR_Count",
            "Full_Spearman",
            "LOO_Spearman",
            "Delta_Spearman"
        ]
    ]
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

influence21b.to_csv(
    CELL_DIR / "stage21b_correlation_influence_full.csv",
    index=False
)

table41.to_csv(
    CELL_DIR / "stage21b_correlation_stability.csv",
    index=False
)

table42.to_csv(
    CELL_DIR / "stage21b_largest_pearson_influences.csv",
    index=False
)

table43.to_csv(
    CELL_DIR / "stage21b_largest_spearman_influences.csv",
    index=False
)

print("\nStage 21B completed successfully.")
print("All 47 cells retained in the primary correlations.")
print("No regression or significance tests performed.")


TABLE 41: CORRELATION STABILITY
       Predictor  Full_Pearson  Min_LOO_Pearson  Max_LOO_Pearson  Full_Spearman  Min_LOO_Spearman  Max_LOO_Spearman
  SL_Trend_mm_yr      -0.09241         -0.17518         -0.02061       -0.11772          -0.18323          -0.06099
SL_Variability_m      -0.00438         -0.17223          0.08040       -0.20907          -0.28548          -0.16559

TABLE 42: LARGEST PEARSON INFLUENCES
       Predictor Removed_Pixel_ID  Valid_WLR_Count  Full_Pearson  LOO_Pearson  Delta_Pearson
  SL_Trend_mm_yr        R001_C003              376      -0.09241     -0.17518       -0.08277
  SL_Trend_mm_yr        R019_C014              348      -0.09241     -0.02061        0.07180
  SL_Trend_mm_yr        R024_C014              160      -0.09241     -0.02306        0.06935
  SL_Trend_mm_yr        R028_C021               94      -0.09241     -0.14572       -0.05331
  SL_Trend_mm_yr        R023_C014              315      -0.09241     -0.04849        0.04391
SL_Variability_m       

In [16]:
# ============================================================
# STAGE 22A: BASELINE CELL-LEVEL OLS REGRESSION
#
# Primary response:
#   Median WLR (m/year)
#
# Models:
#   M1: Sea-level trend
#   M2: Sea-level variability
#   M3: Both predictors
#
# All 47 cells retained.
# Equal cell weighting.
# Predictors standardised using full-sample population SD.
#
# This stage reports descriptive OLS diagnostics.
# Conventional coefficient p-values are not interpreted yet.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------------------
# 1. Prepare aligned analytical dataset
# ------------------------------------------------------------

cells = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    .copy()
)

columns = [
    "WLR_Median",
    "SL_Trend_mm_yr",
    "SL_Variability_m",
    "Valid_WLR_Count"
]

regression_data = cells[columns].apply(
    pd.to_numeric,
    errors="coerce"
)

assert len(regression_data) == 47
assert regression_data.index.is_unique
assert np.isfinite(
    regression_data.to_numpy(dtype=float)
).all()

assert int(
    regression_data["Valid_WLR_Count"].sum()
) == 12452

# ------------------------------------------------------------
# 2. Standardise predictors
# ------------------------------------------------------------

predictor_names = [
    "SL_Trend_mm_yr",
    "SL_Variability_m"
]

standardisation = {}

for predictor in predictor_names:

    mean_value = regression_data[predictor].mean()

    sd_value = regression_data[predictor].std(
        ddof=0
    )

    assert sd_value > 0

    regression_data[
        predictor + "_z"
    ] = (
        regression_data[predictor] - mean_value
    ) / sd_value

    standardisation[predictor] = {
        "Mean": mean_value,
        "Population_SD": sd_value
    }

table44 = pd.DataFrame(
    standardisation
).T

table44.index.name = "Predictor"

print("\nTABLE 44: PREDICTOR STANDARDISATION")

print(
    table44.round(8).to_string()
)

# ------------------------------------------------------------
# 3. Specify and fit models
# ------------------------------------------------------------

model_specs = {
    "M1_Trend": [
        "SL_Trend_mm_yr_z"
    ],
    "M2_Variability": [
        "SL_Variability_m_z"
    ],
    "M3_Combined": [
        "SL_Trend_mm_yr_z",
        "SL_Variability_m_z"
    ]
}

y = regression_data["WLR_Median"]

ols_models = {}

model_records = []
coefficient_records = []

for model_name, predictors in model_specs.items():

    X = sm.add_constant(
        regression_data[predictors],
        has_constant="add"
    )

    model = sm.OLS(
        y,
        X
    ).fit()

    ols_models[model_name] = model

    residuals = np.asarray(
        model.resid,
        dtype=float
    )

    fitted = np.asarray(
        model.fittedvalues,
        dtype=float
    )

    model_records.append({
        "Model": model_name,
        "Cells": int(model.nobs),
        "Predictors": len(predictors),
        "R_Squared": model.rsquared,
        "Adjusted_R_Squared": model.rsquared_adj,
        "RMSE": np.sqrt(
            np.mean(residuals ** 2)
        ),
        "AIC": model.aic,
        "BIC": model.bic,
        "Residual_Mean": residuals.mean(),
        "Residual_SD": residuals.std(ddof=1)
    })

    for term in model.params.index:

        coefficient_records.append({
            "Model": model_name,
            "Term": term,
            "Coefficient": model.params[term],
            "Standard_Error_OLS": model.bse[term],
            "T_Statistic_OLS": model.tvalues[term],
            "P_OLS_Unadjusted": model.pvalues[term]
        })

# ------------------------------------------------------------
# 4. Table 45: Model fit
# ------------------------------------------------------------

table45 = pd.DataFrame(
    model_records
)

print("\nTABLE 45: BASELINE OLS MODEL FIT")

print(
    table45.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 46: Coefficients
# ------------------------------------------------------------

table46 = pd.DataFrame(
    coefficient_records
)

print("\nTABLE 46: OLS COEFFICIENTS")

print(
    table46.round(6).to_string(index=False)
)

print(
    "\nNOTE: OLS coefficient p-values are provisional. "
    "Do not interpret them before residual diagnostics."
)

# ------------------------------------------------------------
# 6. Save fitted values and residuals
# ------------------------------------------------------------

diagnostics = regression_data[
    [
        "WLR_Median",
        "Valid_WLR_Count"
    ]
].copy()

for model_name, model in ols_models.items():

    diagnostics[
        model_name + "_Fitted"
    ] = np.asarray(
        model.fittedvalues
    )

    diagnostics[
        model_name + "_Residual"
    ] = np.asarray(
        model.resid
    )

diagnostics = diagnostics.reset_index()

# ------------------------------------------------------------
# 7. Verify results
# ------------------------------------------------------------

assert len(ols_models) == 3
assert len(diagnostics) == 47

for model_name, model in ols_models.items():

    assert np.isfinite(model.params).all()
    assert np.isfinite(model.resid).all()
    assert model.nobs == 47

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

table44.to_csv(
    CELL_DIR / "stage22a_predictor_standardisation.csv"
)

table45.to_csv(
    CELL_DIR / "stage22a_ols_model_fit.csv",
    index=False
)

table46.to_csv(
    CELL_DIR / "stage22a_ols_coefficients.csv",
    index=False
)

diagnostics.to_csv(
    CELL_DIR / "stage22a_ols_fitted_residuals.csv",
    index=False
)

print("\nStage 22A completed successfully.")
print("Three OLS models fitted using all 47 cells.")
print("Original dataset remains unchanged.")


TABLE 44: PREDICTOR STANDARDISATION
                      Mean  Population_SD
Predictor                                
SL_Trend_mm_yr    3.933072       0.045709
SL_Variability_m  0.027239       0.000404

TABLE 45: BASELINE OLS MODEL FIT
         Model  Cells  Predictors  R_Squared  Adjusted_R_Squared     RMSE        AIC        BIC  Residual_Mean  Residual_SD
      M1_Trend     47           1   0.008539           -0.013493 0.907422 128.248373 131.948668            0.0     0.917233
M2_Variability     47           1   0.000019           -0.022203 0.911313 128.650541 132.350836           -0.0     0.921165
   M3_Combined     47           2   0.009184           -0.035853 0.907127 130.217816 135.768258           -0.0     0.916934

TABLE 46: OLS COEFFICIENTS
         Model               Term  Coefficient  Standard_Error_OLS  T_Statistic_OLS  P_OLS_Unadjusted
      M1_Trend              const     0.017792            0.135271         0.131526          0.895946
      M1_Trend   SL_Trend_mm_yr_z

In [17]:
# ============================================================
# STAGE 22B: OLS REGRESSION DIAGNOSTICS
#
# Models: M1, M2 and M3 from Stage 22A
#
# Diagnostics:
# 1. Residual Moran's I: Queen and Rook
# 2. Breusch-Pagan heteroskedasticity test
# 3. Jarque-Bera residual normality test
# 4. Leverage and Cook's distance
#
# All 47 cells retained.
# No model modifications or exclusions.
# ============================================================

import numpy as np
import pandas as pd

from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

# ------------------------------------------------------------
# 1. Validate existing model objects
# ------------------------------------------------------------

assert len(ols_models) == 3
assert len(cell_ids) == 47

weights = {
    "Queen": np.asarray(queen_W, dtype=float),
    "Rook": np.asarray(rook_W, dtype=float)
}

for name, W in weights.items():

    assert W.shape == (47, 47), name

    assert np.allclose(
        W.sum(axis=1),
        1
    ), name

    assert np.allclose(
        np.diag(W),
        0
    ), name

# ------------------------------------------------------------
# 2. Moran's I and permutation test
# ------------------------------------------------------------

def morans_i_residual(values, W):

    values = np.asarray(
        values,
        dtype=float
    )

    z = values - values.mean()

    denominator = z @ z

    if np.isclose(denominator, 0):
        return np.nan

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / denominator
    )


def residual_moran_test(
    residuals,
    W,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    observed = morans_i_residual(
        residuals,
        W
    )

    simulated = np.empty(
        permutations,
        dtype=float
    )

    for i in range(permutations):

        simulated[i] = morans_i_residual(
            rng.permutation(residuals),
            W
        )

    null_mean = simulated.mean()

    p_value = (
        1 + np.count_nonzero(
            np.abs(simulated - null_mean)
            >= abs(observed - null_mean)
        )
    ) / (permutations + 1)

    return {
        "Residual_Moran_I": observed,
        "Null_Mean": null_mean,
        "Null_SD": simulated.std(ddof=1),
        "P_Two_Sided": p_value
    }

# ------------------------------------------------------------
# 3. Table 47: Residual spatial autocorrelation
# ------------------------------------------------------------

spatial_records = []

for model_name, model in ols_models.items():

    residuals = np.asarray(
        model.resid,
        dtype=float
    )

    assert len(residuals) == 47

    for weights_name, W in weights.items():

        result = residual_moran_test(
            residuals,
            W,
            permutations=9999,
            seed=42
        )

        spatial_records.append({
            "Model": model_name,
            "Weights": weights_name,
            "Cells": 47,
            **result
        })

table47 = pd.DataFrame(
    spatial_records
)

print("\nTABLE 47: RESIDUAL MORAN'S I")

print(
    table47.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Table 48: Heteroskedasticity and normality
# ------------------------------------------------------------

diagnostic_records = []

for model_name, model in ols_models.items():

    residuals = np.asarray(
        model.resid,
        dtype=float
    )

    # Breusch-Pagan test:
    # LM statistic, LM p-value, F statistic, F p-value.
    bp = het_breuschpagan(
        residuals,
        model.model.exog
    )

    # Jarque-Bera test:
    # JB statistic, p-value, skewness, kurtosis.
    jb = jarque_bera(
        residuals
    )

    diagnostic_records.append({
        "Model": model_name,
        "BP_LM": bp[0],
        "BP_P": bp[1],
        "BP_F": bp[2],
        "BP_F_P": bp[3],
        "JB_Statistic": jb[0],
        "JB_P": jb[1],
        "Residual_Skewness": jb[2],
        "Residual_Kurtosis": jb[3]
    })

table48 = pd.DataFrame(
    diagnostic_records
)

print("\nTABLE 48: HETEROSKEDASTICITY AND NORMALITY")

print(
    table48.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Influence diagnostics
# ------------------------------------------------------------

influence_records = []

for model_name, model in ols_models.items():

    influence = model.get_influence()

    leverage = influence.hat_matrix_diag

    cooks_d = influence.cooks_distance[0]

    studentised = (
        influence.resid_studentized_external
    )

    for i, pixel_id in enumerate(cell_ids):

        influence_records.append({
            "Model": model_name,
            "Pixel_ID": pixel_id,
            "Leverage": leverage[i],
            "Cooks_D": cooks_d[i],
            "Studentised_Residual": studentised[i]
        })

influence_table = pd.DataFrame(
    influence_records
)

assert len(influence_table) == 141

# ------------------------------------------------------------
# 6. Table 49: Maximum influence per model
# ------------------------------------------------------------

table49 = (
    influence_table
    .sort_values(
        ["Model", "Cooks_D"],
        ascending=[True, False]
    )
    .groupby(
        "Model",
        group_keys=False
    )
    .head(5)
)

print("\nTABLE 49: FIVE LARGEST COOK'S DISTANCES")

print(
    table49.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Table 50: Influence screening counts
# ------------------------------------------------------------

screening_records = []

for model_name, model in ols_models.items():

    subset = influence_table.loc[
        influence_table["Model"] == model_name
    ]

    n = int(model.nobs)

    # Includes intercept.
    k = int(model.model.exog.shape[1])

    cook_reference = 4 / n

    leverage_reference = 2 * k / n

    screening_records.append({
        "Model": model_name,
        "Cells": n,
        "Parameters_Including_Intercept": k,
        "Cook_Reference_4_over_n": cook_reference,
        "Cells_Above_Cook_Reference": int(
            (
                subset["Cooks_D"]
                > cook_reference
            ).sum()
        ),
        "Leverage_Reference_2k_over_n": leverage_reference,
        "Cells_Above_Leverage_Reference": int(
            (
                subset["Leverage"]
                > leverage_reference
            ).sum()
        ),
        "Cells_Abs_Studentised_Above_3": int(
            (
                subset["Studentised_Residual"].abs()
                > 3
            ).sum()
        )
    })

table50 = pd.DataFrame(
    screening_records
)

print("\nTABLE 50: INFLUENCE SCREENING")

print(
    table50.round(5).to_string(index=False)
)

print(
    "\nNOTE: Influence thresholds are screening "
    "references, not automatic exclusion rules."
)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

table47.to_csv(
    CELL_DIR / "stage22b_residual_moran.csv",
    index=False
)

table48.to_csv(
    CELL_DIR / "stage22b_heteroskedasticity_normality.csv",
    index=False
)

influence_table.to_csv(
    CELL_DIR / "stage22b_influence_all_cells.csv",
    index=False
)

table49.to_csv(
    CELL_DIR / "stage22b_largest_cooks_distance.csv",
    index=False
)

table50.to_csv(
    CELL_DIR / "stage22b_influence_screening.csv",
    index=False
)

print("\nStage 22B completed successfully.")
print("All 47 cells retained.")
print("No regression models modified.")


TABLE 47: RESIDUAL MORAN'S I
         Model Weights  Cells  Residual_Moran_I  Null_Mean  Null_SD  P_Two_Sided
      M1_Trend   Queen     47           0.13097   -0.02215  0.10080       0.1241
      M1_Trend    Rook     47           0.18842   -0.02135  0.12763       0.1053
M2_Variability   Queen     47           0.14343   -0.02205  0.10007       0.0982
M2_Variability    Rook     47           0.20300   -0.02121  0.12667       0.0882
   M3_Combined   Queen     47           0.13572   -0.02221  0.10124       0.1161
   M3_Combined    Rook     47           0.19330   -0.02144  0.12811       0.0996

TABLE 48: HETEROSKEDASTICITY AND NORMALITY
         Model   BP_LM    BP_P    BP_F  BP_F_P  JB_Statistic  JB_P  Residual_Skewness  Residual_Kurtosis
      M1_Trend 0.53575 0.46420 0.51886 0.47505     205.13851   0.0            1.64256           12.69327
M2_Variability 0.14560 0.70278 0.13984 0.71020     225.71126   0.0            1.71301           13.17444
   M3_Combined 0.73526 0.69237 0.34964 0.706

In [18]:
# ============================================================
# STAGE 22C: LEAVE-ONE-CELL-OUT OLS SENSITIVITY
#
# Models: M1, M2 and M3
# Response: median WLR (m/year)
#
# IMPORTANT:
# Predictors retain the FULL-SAMPLE standardisation from
# Stage 22A so coefficients remain directly comparable.
#
# All 47 cells remain in the primary models.
# No significance tests or permanent exclusions.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------------------
# 1. Validate Stage 22A objects
# ------------------------------------------------------------

assert len(regression_data) == 47
assert len(ols_models) == 3
assert regression_data.index.is_unique

assert list(regression_data.index) == list(cell_ids)

assert np.isfinite(
    regression_data[
        [
            "WLR_Median",
            "SL_Trend_mm_yr_z",
            "SL_Variability_m_z"
        ]
    ].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# 2. Leave-one-cell-out model fitting
# ------------------------------------------------------------

loo_records = []

for model_name, predictors in model_specs.items():

    full_model = ols_models[model_name]

    for removed_id in cell_ids:

        subset = regression_data.drop(
            index=removed_id
        )

        X = sm.add_constant(
            subset[predictors],
            has_constant="add"
        )

        y = subset["WLR_Median"]

        loo_model = sm.OLS(
            y,
            X
        ).fit()

        record = {
            "Model": model_name,
            "Removed_Pixel_ID": removed_id,
            "Cells": int(loo_model.nobs),
            "Full_R2": full_model.rsquared,
            "LOO_R2": loo_model.rsquared,
            "Delta_R2": (
                loo_model.rsquared
                - full_model.rsquared
            ),
            "Full_RMSE": np.sqrt(
                np.mean(
                    np.asarray(
                        full_model.resid
                    ) ** 2
                )
            ),
            "LOO_RMSE": np.sqrt(
                np.mean(
                    np.asarray(
                        loo_model.resid
                    ) ** 2
                )
            )
        }

        for term in full_model.params.index:

            full_beta = full_model.params[term]

            loo_beta = loo_model.params[term]

            record[
                "Full_Beta_" + term
            ] = full_beta

            record[
                "LOO_Beta_" + term
            ] = loo_beta

            record[
                "Delta_Beta_" + term
            ] = loo_beta - full_beta

        loo_records.append(record)

loo22c = pd.DataFrame(
    loo_records
)

assert len(loo22c) == 141
assert (loo22c["Cells"] == 46).all()

# ------------------------------------------------------------
# 3. Table 51: R-squared sensitivity
# ------------------------------------------------------------

table51 = (
    loo22c
    .groupby(
        "Model",
        as_index=False
    )
    .agg(
        Full_R2=("Full_R2", "first"),
        Min_LOO_R2=("LOO_R2", "min"),
        Max_LOO_R2=("LOO_R2", "max"),
        Min_Delta_R2=("Delta_R2", "min"),
        Max_Delta_R2=("Delta_R2", "max"),
        Full_RMSE=("Full_RMSE", "first"),
        Min_LOO_RMSE=("LOO_RMSE", "min"),
        Max_LOO_RMSE=("LOO_RMSE", "max")
    )
)

print("\nTABLE 51: MODEL FIT SENSITIVITY")

print(
    table51.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Table 52: Coefficient sensitivity
# ------------------------------------------------------------

coefficient_records = []

for model_name, predictors in model_specs.items():

    subset = loo22c.loc[
        loo22c["Model"] == model_name
    ]

    for term in predictors:

        full_col = "Full_Beta_" + term
        loo_col = "LOO_Beta_" + term

        full_beta = subset[full_col].iloc[0]

        loo_values = subset[loo_col]

        coefficient_records.append({
            "Model": model_name,
            "Term": term,
            "Full_Beta": full_beta,
            "Min_LOO_Beta": loo_values.min(),
            "Max_LOO_Beta": loo_values.max(),
            "LOO_Sign_Reversals": int(
                (
                    np.sign(loo_values)
                    != np.sign(full_beta)
                ).sum()
            )
        })

table52 = pd.DataFrame(
    coefficient_records
)

print("\nTABLE 52: COEFFICIENT SENSITIVITY")

print(
    table52.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 53: Largest R-squared changes
# ------------------------------------------------------------

loo22c["Abs_Delta_R2"] = (
    loo22c["Delta_R2"].abs()
)

table53 = (
    loo22c
    .sort_values(
        [
            "Model",
            "Abs_Delta_R2"
        ],
        ascending=[True, False]
    )
    .groupby(
        "Model",
        group_keys=False
    )
    .head(5)
)

print("\nTABLE 53: LARGEST R-SQUARED CHANGES")

print(
    table53[
        [
            "Model",
            "Removed_Pixel_ID",
            "Full_R2",
            "LOO_R2",
            "Delta_R2",
            "Full_RMSE",
            "LOO_RMSE"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Table 54: Combined-model focus cells
# ------------------------------------------------------------

focus_ids = [
    "R001_C003",
    "R019_C014",
    "R001_C005",
    "R024_C014",
    "R023_C014"
]

combined = loo22c.loc[
    (loo22c["Model"] == "M3_Combined")
    & (
        loo22c["Removed_Pixel_ID"].isin(
            focus_ids
        )
    )
].copy()

table54 = combined[
    [
        "Removed_Pixel_ID",
        "LOO_R2",
        "Delta_R2",
        "LOO_Beta_SL_Trend_mm_yr_z",
        "LOO_Beta_SL_Variability_m_z"
    ]
]

print("\nTABLE 54: COMBINED MODEL INFLUENCE")

print(
    table54.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

loo22c.to_csv(
    CELL_DIR / "stage22c_leave_one_cell_out_full.csv",
    index=False
)

table51.to_csv(
    CELL_DIR / "stage22c_model_fit_sensitivity.csv",
    index=False
)

table52.to_csv(
    CELL_DIR / "stage22c_coefficient_sensitivity.csv",
    index=False
)

table53.to_csv(
    CELL_DIR / "stage22c_largest_r2_changes.csv",
    index=False
)

table54.to_csv(
    CELL_DIR / "stage22c_combined_model_focus_cells.csv",
    index=False
)

print("\nStage 22C completed successfully.")
print("Primary models retain all 47 cells.")
print("Leave-one-out fits used only for sensitivity analysis.")


TABLE 51: MODEL FIT SENSITIVITY
         Model  Full_R2  Min_LOO_R2  Max_LOO_R2  Min_Delta_R2  Max_Delta_R2  Full_RMSE  Min_LOO_RMSE  Max_LOO_RMSE
      M1_Trend 0.008539    0.000425    0.030688     -0.008115      0.022148   0.907422      0.652017      0.917233
M2_Variability 0.000019    0.000001    0.029662     -0.000018      0.029643   0.911313      0.645923      0.921165
   M3_Combined 0.009184    0.003637    0.048177     -0.005546      0.038993   0.907127      0.645699      0.916934

TABLE 52: COEFFICIENT SENSITIVITY
         Model               Term  Full_Beta  Min_LOO_Beta  Max_LOO_Beta  LOO_Sign_Reversals
      M1_Trend   SL_Trend_mm_yr_z  -0.084214     -0.153082     -0.013376                   0
M2_Variability SL_Variability_m_z  -0.003988     -0.162159      0.074293                  11
   M3_Combined   SL_Trend_mm_yr_z  -0.091720     -0.149858      0.017904                   1
   M3_Combined SL_Variability_m_z   0.024321     -0.128696      0.121258                   2

TABLE 

In [19]:
# ============================================================
# STAGE 22D: HC3 ROBUST INFERENCE AND COLLINEARITY AUDIT
#
# Existing models: M1, M2, M3
# Primary response: median WLR
#
# Purpose:
# 1. Compare conventional and HC3 standard errors.
# 2. Report HC3 confidence intervals.
# 3. Check combined-model predictor collinearity.
#
# No model refitting with exclusions.
# No spatial-dependence correction.
# All 47 cells retained.
# ============================================================

import numpy as np
import pandas as pd

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

# ------------------------------------------------------------
# 1. Validate existing models
# ------------------------------------------------------------

assert len(ols_models) == 3

for model_name, model in ols_models.items():
    assert int(model.nobs) == 47
    assert np.isfinite(model.params).all()

# ------------------------------------------------------------
# 2. HC3 robust covariance results
# ------------------------------------------------------------

robust_records = []

for model_name, model in ols_models.items():

    robust = model.get_robustcov_results(
        cov_type="HC3",
        use_t=True
    )

    terms = list(model.params.index)

    ci = robust.conf_int(alpha=0.05)

    for i, term in enumerate(terms):

        robust_records.append({
            "Model": model_name,
            "Term": term,
            "Coefficient": float(model.params.iloc[i]),
            "SE_Conventional": float(model.bse.iloc[i]),
            "SE_HC3": float(robust.bse[i]),
            "HC3_to_OLS_SE_Ratio": (
                float(robust.bse[i])
                / float(model.bse.iloc[i])
            ),
            "P_Conventional": float(
                model.pvalues.iloc[i]
            ),
            "P_HC3": float(robust.pvalues[i]),
            "HC3_CI_Lower": float(ci[i, 0]),
            "HC3_CI_Upper": float(ci[i, 1])
        })

table55 = pd.DataFrame(robust_records)

expected_coefficients = sum(
    len(model.params)
    for model in ols_models.values()
)

assert expected_coefficients == 7
assert len(table55) == expected_coefficients

print("\nTABLE 55: CONVENTIONAL VS HC3 INFERENCE")

print(
    table55.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 3. Check combined-model multicollinearity
# ------------------------------------------------------------

combined_model = ols_models["M3_Combined"]

X = np.asarray(
    combined_model.model.exog,
    dtype=float
)

terms = list(
    combined_model.model.exog_names
)

assert X.shape == (47, 3)

vif_records = []

for i, term in enumerate(terms):

    if term == "const":
        continue

    vif_records.append({
        "Term": term,
        "VIF": float(
            variance_inflation_factor(X, i)
        )
    })

table56 = pd.DataFrame(vif_records)

print("\nTABLE 56: COMBINED-MODEL VIF")

print(
    table56.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Table 57: HC3 summary for slope coefficients
# ------------------------------------------------------------

slopes = table55.loc[
    table55["Term"] != "const"
].copy()

table57 = slopes[
    [
        "Model",
        "Term",
        "Coefficient",
        "SE_HC3",
        "P_HC3",
        "HC3_CI_Lower",
        "HC3_CI_Upper"
    ]
]

print("\nTABLE 57: HC3 SLOPE SUMMARY")

print(
    table57.round(6).to_string(index=False)
)

print(
    "\nInterpretation safeguard:"
    "\nHC3 addresses heteroskedasticity and leverage "
    "in covariance estimation."
    "\nIt does not correct spatial dependence, "
    "non-normality or temporal mismatch."
)

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------

table55.to_csv(
    CELL_DIR / "stage22d_conventional_vs_hc3.csv",
    index=False
)

table56.to_csv(
    CELL_DIR / "stage22d_combined_model_vif.csv",
    index=False
)

table57.to_csv(
    CELL_DIR / "stage22d_hc3_slope_summary.csv",
    index=False
)

print("\nStage 22D completed successfully.")
print("All 47 cells retained.")
print("No observations excluded or models replaced.")


TABLE 55: CONVENTIONAL VS HC3 INFERENCE
         Model               Term  Coefficient  SE_Conventional   SE_HC3  HC3_to_OLS_SE_Ratio  P_Conventional    P_HC3  HC3_CI_Lower  HC3_CI_Upper
      M1_Trend              const     0.017792         0.135271 0.138558             1.024306        0.895946 0.898401     -0.261279      0.296863
      M1_Trend   SL_Trend_mm_yr_z    -0.084214         0.135271 0.146243             1.081112        0.536718 0.567587     -0.378761      0.210334
M2_Variability              const     0.017792         0.135851 0.142447             1.048556        0.896387 0.901159     -0.269111      0.304694
M2_Variability SL_Variability_m_z    -0.003988         0.135851 0.212358             1.563172        0.976711 0.985100     -0.431698      0.423722
   M3_Combined              const     0.017792         0.136755 0.145514             1.064050        0.897082 0.903244     -0.275472      0.311055
   M3_Combined   SL_Trend_mm_yr_z    -0.091720         0.143774 0.160012     

In [20]:
# ============================================================
# STAGE 22E: ROOK RESIDUAL MORAN PERMUTATION SENSITIVITY
#
# Compare:
# 1. Unrestricted permutations
# 2. Permutations restricted to Rook components
#
# Models: M1, M2, M3
# All 47 cells retained.
# No regression models modified.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate alignment and Rook components
# ------------------------------------------------------------

assert len(cell_ids) == 47
assert len(rook_components) == 3

component_sizes = sorted(
    [len(component) for component in rook_components],
    reverse=True
)

assert component_sizes == [38, 5, 4]

id_to_position = {
    pixel_id: i
    for i, pixel_id in enumerate(cell_ids)
}

component_positions = [
    np.array(
        [id_to_position[pixel_id] for pixel_id in component],
        dtype=int
    )
    for component in rook_components
]

all_positions = np.concatenate(component_positions)

assert len(all_positions) == 47
assert len(np.unique(all_positions)) == 47
assert set(all_positions) == set(range(47))

W = np.asarray(rook_W, dtype=float)

assert W.shape == (47, 47)
assert np.allclose(W.sum(axis=1), 1)

# ------------------------------------------------------------
# 2. Moran's I
# ------------------------------------------------------------

def stage22e_moran(values, W):

    values = np.asarray(values, dtype=float)

    z = values - values.mean()

    denominator = z @ z

    if denominator <= 0:
        raise ValueError("Zero residual variance.")

    return (
        len(values) / W.sum()
        * (z @ W @ z)
        / denominator
    )

# ------------------------------------------------------------
# 3. Permutation test
# ------------------------------------------------------------

def stage22e_permutation_test(
    values,
    W,
    restricted=False,
    permutations=9999,
    seed=42
):

    rng = np.random.default_rng(seed)

    values = np.asarray(values, dtype=float)

    observed = stage22e_moran(values, W)

    simulated = np.empty(
        permutations,
        dtype=float
    )

    for iteration in range(permutations):

        if restricted:

            shuffled = values.copy()

            for positions in component_positions:

                shuffled[positions] = rng.permutation(
                    values[positions]
                )

        else:

            shuffled = rng.permutation(values)

        simulated[iteration] = stage22e_moran(
            shuffled,
            W
        )

    null_mean = simulated.mean()

    p_value = (
        1 + np.count_nonzero(
            np.abs(simulated - null_mean)
            >= abs(observed - null_mean)
        )
    ) / (permutations + 1)

    return {
        "Moran_I": observed,
        "Null_Mean": null_mean,
        "Null_SD": simulated.std(ddof=1),
        "P_Two_Sided": p_value
    }

# ------------------------------------------------------------
# 4. Run all three models
# ------------------------------------------------------------

records = []

for model_name, model in ols_models.items():

    residuals = np.asarray(
        model.resid,
        dtype=float
    )

    assert len(residuals) == 47

    for restricted, method in [
        (False, "Unrestricted"),
        (True, "Component-restricted")
    ]:

        result = stage22e_permutation_test(
            residuals,
            W,
            restricted=restricted,
            permutations=9999,
            seed=42
        )

        records.append({
            "Model": model_name,
            "Weights": "Rook",
            "Permutation_Method": method,
            "Cells": 47,
            "Components": 3,
            **result
        })

table58 = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. Table 58: Residual Moran comparison
# ------------------------------------------------------------

print("\nTABLE 58: ROOK RESIDUAL MORAN SENSITIVITY")

print(
    table58.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Table 59: Difference between null assumptions
# ------------------------------------------------------------

unrestricted = (
    table58.loc[
        table58["Permutation_Method"] == "Unrestricted"
    ]
    .set_index("Model")
)

restricted = (
    table58.loc[
        table58["Permutation_Method"]
        == "Component-restricted"
    ]
    .set_index("Model")
)

table59 = pd.DataFrame({
    "Model": unrestricted.index,
    "Moran_I": unrestricted["Moran_I"].to_numpy(),
    "P_Unrestricted": unrestricted[
        "P_Two_Sided"
    ].to_numpy(),
    "P_Component_Restricted": restricted[
        "P_Two_Sided"
    ].to_numpy(),
    "Difference_In_P": (
        restricted["P_Two_Sided"].to_numpy()
        - unrestricted["P_Two_Sided"].to_numpy()
    )
})

assert np.allclose(
    unrestricted["Moran_I"].to_numpy(),
    restricted["Moran_I"].to_numpy()
)

print("\nTABLE 59: NULL-ASSUMPTION COMPARISON")

print(
    table59.round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

table58.to_csv(
    CELL_DIR / "stage22e_rook_residual_moran_sensitivity.csv",
    index=False
)

table59.to_csv(
    CELL_DIR / "stage22e_null_assumption_comparison.csv",
    index=False
)

print("\nStage 22E completed successfully.")
print("All 47 cells retained.")
print("No regression models modified.")


TABLE 58: ROOK RESIDUAL MORAN SENSITIVITY
         Model Weights   Permutation_Method  Cells  Components  Moran_I  Null_Mean  Null_SD  P_Two_Sided
      M1_Trend    Rook         Unrestricted     47           3  0.18842   -0.02135  0.12763       0.1053
      M1_Trend    Rook Component-restricted     47           3  0.18842   -0.08869  0.12094       0.0242
M2_Variability    Rook         Unrestricted     47           3  0.20300   -0.02121  0.12667       0.0882
M2_Variability    Rook Component-restricted     47           3  0.20300   -0.08764  0.12046       0.0206
   M3_Combined    Rook         Unrestricted     47           3  0.19330   -0.02144  0.12811       0.0996
   M3_Combined    Rook Component-restricted     47           3  0.19330   -0.08685  0.12163       0.0234

TABLE 59: NULL-ASSUMPTION COMPARISON
         Model  Moran_I  P_Unrestricted  P_Component_Restricted  Difference_In_P
      M1_Trend  0.18842          0.1053                  0.0242          -0.0811
M2_Variability  0.2030

In [21]:
# ============================================================
# STAGE 22F: ROOK COMPONENT STRUCTURE AUDIT
#
# Purpose:
# 1. Verify Rook component membership.
# 2. Summarise response and predictors by component.
# 3. Examine M3 residual means by component.
# 4. Separate within-component and between-component
#    descriptive variation.
#
# No new regression models.
# No significance tests.
# All 47 cells retained.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Build component membership
# ------------------------------------------------------------

assert len(rook_components) == 3
assert len(cell_ids) == 47

components_sorted = sorted(
    rook_components,
    key=len,
    reverse=True
)

membership = {}

for component_number, component in enumerate(
    components_sorted,
    start=1
):
    for pixel_id in component:

        assert pixel_id not in membership

        membership[pixel_id] = component_number

assert set(membership) == set(cell_ids)

# ------------------------------------------------------------
# 2. Prepare aligned data
# ------------------------------------------------------------

audit = (
    analysis_grid
    .set_index("Pixel_ID")
    .loc[cell_ids]
    [
        [
            "WLR_Median",
            "WLR_Mean",
            "SL_Trend_mm_yr",
            "SL_Variability_m",
            "Valid_WLR_Count"
        ]
    ]
    .copy()
)

audit["Rook_Component"] = [
    membership[pixel_id]
    for pixel_id in audit.index
]

# Assign M3 residuals by Pixel_ID, not by assumed position.

m3_residuals = pd.Series(
    np.asarray(
        ols_models["M3_Combined"].resid,
        dtype=float
    ),
    index=regression_data.index
)

audit["M3_Residual"] = m3_residuals.reindex(
    audit.index
)

assert len(audit) == 47

assert np.isfinite(
    audit[
        [
            "WLR_Median",
            "WLR_Mean",
            "SL_Trend_mm_yr",
            "SL_Variability_m",
            "Valid_WLR_Count",
            "M3_Residual"
        ]
    ].to_numpy(dtype=float)
).all()

assert (
    audit.groupby("Rook_Component")
    .size()
    .sort_values(ascending=False)
    .tolist()
    == [38, 5, 4]
)

# ------------------------------------------------------------
# 3. Table 60: Component-level characteristics
# ------------------------------------------------------------

table60 = (
    audit
    .groupby("Rook_Component")
    .agg(
        Cells=("WLR_Median", "size"),
        Valid_Transects=("Valid_WLR_Count", "sum"),
        WLR_Median_Mean=("WLR_Median", "mean"),
        WLR_Median_Median=("WLR_Median", "median"),
        WLR_Median_SD=("WLR_Median", "std"),
        SL_Trend_Mean=("SL_Trend_mm_yr", "mean"),
        SL_Trend_SD=("SL_Trend_mm_yr", "std"),
        SL_Variability_Mean=("SL_Variability_m", "mean"),
        SL_Variability_SD=("SL_Variability_m", "std"),
        M3_Residual_Mean=("M3_Residual", "mean"),
        M3_Residual_SD=("M3_Residual", "std")
    )
    .reset_index()
)

print("\nTABLE 60: ROOK COMPONENT CHARACTERISTICS")

print(
    table60.round(7).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Variance decomposition
#
# Total SS = Within-component SS + Between-component SS
#
# These are descriptive proportions, not model R-squared.
# ------------------------------------------------------------

decomposition_records = []

variables = [
    "WLR_Median",
    "SL_Trend_mm_yr",
    "SL_Variability_m",
    "M3_Residual"
]

for variable in variables:

    values = audit[variable]

    grand_mean = values.mean()

    total_ss = float(
        ((values - grand_mean) ** 2).sum()
    )

    group_means = (
        audit.groupby("Rook_Component")[variable]
        .transform("mean")
    )

    within_ss = float(
        ((values - group_means) ** 2).sum()
    )

    between_ss = float(
        ((group_means - grand_mean) ** 2).sum()
    )

    assert np.isclose(
        total_ss,
        within_ss + between_ss,
        rtol=1e-9,
        atol=1e-10
    )

    decomposition_records.append({
        "Variable": variable,
        "Total_SS": total_ss,
        "Within_Component_SS": within_ss,
        "Between_Component_SS": between_ss,
        "Between_Component_Percent": (
            100 * between_ss / total_ss
            if total_ss > 0
            else np.nan
        )
    })

table61 = pd.DataFrame(
    decomposition_records
)

print("\nTABLE 61: COMPONENT VARIANCE DECOMPOSITION")

print(
    table61.round(7).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Table 62: Component membership
# ------------------------------------------------------------

table62 = (
    audit
    .reset_index()
    [
        [
            "Pixel_ID",
            "Rook_Component",
            "Valid_WLR_Count",
            "WLR_Median",
            "SL_Trend_mm_yr",
            "SL_Variability_m",
            "M3_Residual"
        ]
    ]
    .sort_values(
        ["Rook_Component", "Pixel_ID"]
    )
)

print("\nTABLE 62: COMPONENT MEMBERSHIP")

print(
    table62.round(7).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

table60.to_csv(
    CELL_DIR / "stage22f_component_characteristics.csv",
    index=False
)

table61.to_csv(
    CELL_DIR / "stage22f_component_variance_decomposition.csv",
    index=False
)

table62.to_csv(
    CELL_DIR / "stage22f_component_membership.csv",
    index=False
)

print("\nStage 22F completed successfully.")
print("All 47 cells retained.")
print("No new regression or significance tests performed.")


TABLE 60: ROOK COMPONENT CHARACTERISTICS
 Rook_Component  Cells  Valid_Transects  WLR_Median_Mean  WLR_Median_Median  WLR_Median_SD  SL_Trend_Mean  SL_Trend_SD  SL_Variability_Mean  SL_Variability_SD  M3_Residual_Mean  M3_Residual_SD
              1     38             9569        -0.001426           0.027374       0.914125       3.930990     0.048530             0.027368           0.000278         -0.031156        0.901103
              2      5             1407         0.152346          -0.012089       0.377921       3.962582     0.021138             0.027023           0.000246          0.206814        0.333742
              3      4             1476         0.032169           0.308088       1.574702       3.915960     0.034544             0.026286           0.000150          0.037465        1.625832

TABLE 61: COMPONENT VARIANCE DECOMPOSITION
        Variable  Total_SS  Within_Component_SS  Between_Component_SS  Between_Component_Percent
      WLR_Median 39.033836            38.9284